### <center> **Midterm Analysis Code | Tyler Hilbert | May 7, 2026** </center>

#### <center>**About This File**</center>

The code presented in this notebook is made to generate dataframes and figures that permit analysis of midterm grade data and the effectiveness of advisor intervention efforts. The figures created can be split into two categories:
1. **Tables** - Provide a snapshot of the most current term's midterm grade data and a list of low-performing students who may need contacted for intervention efforts.
2. **Charts** - Allows for a comparative analysis of midterm grade data and intervention efforts over time - either by highlighting "pain point" courses for students or how studetns grades changed from midterm to finals.

In order for a course to be captured on this list, they must meet all of the following:
1. Be listed as a required course for an undergraduate major in the CAED-CCI-CotA hub
2. Be required in the first four semesters (as defined by the current KSU catalog)
3. Require midterm grades be entered per university policy (courses at the 10000 or 20000 level)

Data is sourced from Cognos report 100c. Grade data has been anonymized to maintain student confidentiality and does not reflect the performance of the represented units and their students. For Fall 2025 and earlier, grade data is gathered well after midterm and final grades were entered. Future iterations of the code will involve grade collection immediately upon their posting to Banner.

For the flow of this file, a background of the code including driving questions and audience is discussed first. This is followed by the code that analyzes the data, generates figures (tables/charts). The file closes by discussing the limitations, areas of improvement and planned future implementation of this file.

##### <center>*Driving Questions/Factors*</center>

Prior to this assignment, midterm grade data was manually collected and cleaned before being provided to advisors in an Excel document format. This project aims to streamline that process by quickly cleaning the data and providing midterm grade data to unit leadership for review and action.

In CCI, the advising team has historically reached out to students who were underperforming at midterms to help boost retention rates. However, no efforts have been made to track the number of interventions over time or how effective they may have been in improving student performance. The figures created by this file aim to allow for that analysis, as well as snapshots of the current term's midterm grades.

##### <center>*Target Audience*</center>

The target audience for these figures is leadership in the CAED-CCI-CotA hub. The leadership in these units regularly communicate student performance data to faculty, and these figures aim to provide a quick reference point for leadership to look toward prior to communicating out to faculty. The information from these figures can help identify "pain points" that could be addressed by the faculty to improve student performance.

Additionally, academic advisors in the hub can utilize this information to reflect on their advising efforts and highlight the effectiveness these efforts may have had on final grade data. This can also help inform future interventions to be targeted to key courses or what kinds of resources are effective at improving student performance.

##### <center>*Figure Style Guide*</center>

Originally, I was going to give each college their own color scheme. The logic behind this was that it would be easy to immediately tell which unit you were looking at at a glance. However, I realized this gave the dashboard a more cluttered look, and it would be more consistent to follow a single theme.

As the grade data is looking at a specific institution, the institution's style guide will be followed in the creation of the figures. The core elements of this are as follows:
- Utilizing Blue (#003976) and Gold (#EFAB00) for the core colors of the figures.
- For secondary colors (i.e. alternating row colors), a lighter shade of gold is used.
- Utilizing Segoe UI for the font. 
- For biological relevance, four additional, lighter colors are utilized throughout the figure:
  - Green - Highlighting instances of passing grades/positive outcomes
  - Yellow - Highlighting instances of hazard signs (i.e. minor intervention, at risk)
  - Red - Highlighting instances of negative grades/negative outcomes
  - Gray - For items that are non-applicable (i.e. no intervention needed, no grade reported)

*A quick notes on comments in cells - for repeating figures (i.e. figures that look the same, just different data) only the **first** instance of the cell will have comments. Future cells will not retain those since they follow the same logic*

#### <center> **Section 0 - Importing Libraries and Defining Variables** </center>

This section is dedicated to importing libraries that will be used throughout the code. Additionally, universal variables will be referenced here. These variables will be referenced throughout the code, and are the code that will change most regularly (most recent semester, major codes, etc.). Notes will be made next to variables that need changed and about what time they will need updated. This will help avoid having to sift through the code later to change variables.

In [396]:
#Section 0a - Importing Libraries
import pandas as pd #allows for data manipulation and management of dataframes
import numpy as np #allows for additional manipulation logic of data (reading lists, if statements, etc.)
import plotly.graph_objects as go #allows for the creation of figures and visualizations
import plotly.io as pio #allows for conversion of plotly code to HTML file for uploading to PowerBI
import json #allows for figures to be converted to JSON file for entry into HTML code

In [397]:
#Section 0b - Universal Variables

#The Current Term Map
currentterm = [202610] #IMPORTANT - UPDATE FOR EACH NEW CYCLE
termmap = {10: "Spring", 80: "Fall"} #Reads the last two digits to determine term used.
year = str(currentterm[0])[:4] #reads the first 4 digits to determine the term
termsuffix = int(str(currentterm[0])[4:]) #reads the last two digits to determine the semester
prettyterm = f"{termmap.get(termsuffix, 'Semester')} {year}" #This uses the above to spit out the term that is being examined in a readable format

#The Historical Term Map
figtermmap = { #Built up to Spring 2030 - feel free to add/remove terms as appropriate
    202280 : "Fall 2022",
    202310 : "Spring 2023",
    202380 : "Fall 2023",
    202410 : "Spring 2024",
    202480 : "Fall 2024",
    202510 : "Spring 2025",
    202580 : "Fall 2025",
    202610 : "Spring 2026",
    202680 : "Fall 2026",
    202710 : "Spring 2027",
    202780 : "Fall 2027",
    202810 : "Spring 2028",
    202880 : "Fall 2028",
    202910 : "Spring 2029",
    202980 : "Fall 2029",
    203010 : "Spring 2030"
}

#Heat Term Map List
#IMPORTANT - UPDATE TO MOST RECENT TERM THAT DOES NOT HAVE FINAL GRADES
currentheatterm = [202610]

In [398]:
#Section 0c - Major Variables
#Contains Variables that dictate all the major(s) in a college
caedmajor = ["ID", "ARCH", "COMA", "ARCS"] #Majors in CAED
ccimajor = ["COMM", "DMP", "VCD", "JNL", "EMAT", "ADV", "PR", "PHOT", "APMD","UXDE"] #Majors in CCI
cotamajor = ["FM", "ARTH", "FD", "DNST", "TDTP", "SART", "THEA", "ARTE", "MUST", "MUS", "MUED", "MUT","DANC"] #Majors in CotA
othermajor = ["OTH"]

In [399]:
#Section 0d - Subject Variables
#Contains variables that dictate all the subject(s) in a college
caedsubject = ["AED", "ARCH", "ARCS", "CMGT", "ID"] #Subjects in CAED
ccisubject = ["CCI", "COMM", "EMAT", "MDJ", "VCD"] #Subjects in CCI
cotasubject = ["ART", "ARTH", "ARTS", "DAN", "FDM", "MUS", "THEA"] #Subjects in CotA

In [400]:
#Section 0e - Intervention Variables
#Contains variables that dictate if interventions are needed based on midterm grades
pre202610frsopassgrade = ["A", "A-", "B+","B","B-","C+","S"]
pre202610frsofailgrade = ["C", "C-", "D+", "D", "F","U","SF"]
post202610frsopassgrade = ["A", "A-", "B+","B","B-","C+","C", "C-","S"]
post202610frsofailgrade = ["D+", "D", "F","U", "SF"]
jrsrpassgrade = ["A", "A-", "B+","B","B-","C+","C","C-","D+","D","S"]
jrsrfailgrade = ["F","U","SF"]
dropgrade = ["DR"]
wgrade = ["W"]
nfgrade = ["NF"]
frsocheck = ["FR", "SO"]
jrsrcheck = ["JR", "SR", "GR"]
pre202610terms = [202280, 202310, 202380, 202410, 202480, 202510, 202580]
post202610terms = [202610, 202680, 202710, 202780, 202810, 202880, 202910, 202980, 203010]

In [401]:
#Section 0f - Summary Table Variables
#Contains variables that help build the summary tables

#Outlining what is a pass v. fail midterm grade
mtpass = ["A","A-","B+","B","B-","C+","C","S"]
mtfail = ["C-","D+","D","F","NF","SF","U","W"]

#Defining columns for MT Grades
gradecols = ["MT C or Higher", "MT C-, D, F, W", "MT Not Reported", "Total Grades"]

#Defining header and value variables
headers = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]
values = ["Course Code", "Total Grades", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]

In [402]:
#Section 0g - Heat Map Variables
#Contains variables that help build the heat maps

#Creating the color map that will be used for the heat map
civicustom = [
    [0.0,"#013271"],
    [0.1,"#5E636E"],
    [0.25,"#9D9576"],
    [1.0,"#E7D150"]
]

#Creating a Grade Scale Variable
gradescalenum = [0.0, 1.0, 1.3, 1.7, 2.0, 2.3, 2.7, 3.0, 3.3, 3.7, 4.0]
gradescalestring = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"]

#### <center> **Section 1 - Dataframes** </center>

This section is dedicated to creating the various dataframes that will be used for creating figures. By the end of this section, multiple dataframes will be created:
- A dataframe of midterm grades for all enrolled students (Summary Table, Summary Over Time Line Chart)
- A dataframe of midterm and final grades for hub-only students (Heat Map)
- A dataframe of hub only students who require advisor intervention (Intervention Table, Advisor Contact Excel Sheet, Intervention Over Time Line Chart)

**Please Note:** The code in this file is focused on creating the dataframe only - code for reading the dataframe (i.e. checking shape, size, keys) are not included. Review of the data can be performed in a separate notebook or in Excel.

##### <center>*Loading the Data File*</center>

As touched on previously, the data file used is anonymized grade data from the university system. 

In [403]:
#Section 1a - Loading Data File
mtfull = pd.read_csv("mtgradesanon.csv") #Ensure that file name aligns with what is written
mtfull.head()

,Unnamed: 0,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class
0,0,RW,AED,22860,KC,A-,B,ARCH,ID,202280,FR
1,1,RW,AED,22860,KC,A,A,ARCH,ID,202280,FR
2,2,RW,AED,22860,KC,A,A,ARCH,OTH,202280,JR
3,3,RW,AED,22860,KC,F,B,ARCH,ID,202280,SO
4,4,RW,AED,22860,KC,B,B,ARCH,ID,202280,FR


##### <center>*Removing Unnamed: 0*</center>

This step can be cut in the final version (Unnamed: 0 column is a artifact of anonymizing the data)

In [404]:
#Section 1b - Renaming Unnamed: 0
mtfull = mtfull.rename(columns = {"Unnamed: 0": "Record ID"})
mtfull.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class
0,0,RW,AED,22860,KC,A-,B,ARCH,ID,202280,FR
1,1,RW,AED,22860,KC,A,A,ARCH,ID,202280,FR
2,2,RW,AED,22860,KC,A,A,ARCH,OTH,202280,JR
3,3,RW,AED,22860,KC,F,B,ARCH,ID,202280,SO
4,4,RW,AED,22860,KC,B,B,ARCH,ID,202280,FR


##### <center>*Course Code Columns*</center>

The course code is a concatanated instance of the Subject and Course number. The course code are what courses are commonly called out when discussing courses. This is what will be referenced in figures when calling out specific courses.

In [405]:
#Section 1c - Creating a Course Code Column
mtfull["Course"] = mtfull["Course"].astype(str)
mtfull["Course Code"] = mtfull["Subject"] + " " + mtfull["Course"]
mtfull.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code
0,0,RW,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860
1,1,RW,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860
2,2,RW,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860
3,3,RW,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860
4,4,RW,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860


##### <center>*Registration Code Mapping*</center>

The original file uses the registration codes from Banner. These codes are not obvious what they are, so "decoding" them via mapping helps users better understand the registration status. Additionally, many codes are referencing the same status so this helps the readability with the file.

In [406]:
#Section 1d - Pulling Banner Registration Status Codes
#Note - there may be new codes. If so, review Banner documentation to identify translation.
regcode = mtfull["Registration Status"].unique()
regcode

<StringArray>
['RW', 'RE', 'WW', 'DD', 'W8', 'ND', 'R2', 'SF', 'B1', 'WD', 'NF', 'RA', 'AW',
 'DR', 'B5']
Length: 15, dtype: str

In [407]:
#Section 1e - Creating Reg Code Map
#Having a strict assignment helps ensure codes are assigned appropriately
#Note - if a new registration code is present, need to add it to this list with the full translation
regcodemap = {
    "RW": "Registered",
    "RE": "Registered",
    "WW": "Std Withdrawn",
    "DD": "Admin Dropped",
    "W8": "Std Dropped",
    "ND": "Admin Dropped",
    "R2": "Registered",
    "SF": "Stopped Attending - Failed",
    "B1": "Admin Withdrawn",
    "WD": "Std Withdrawn",
    "NF": "Never Attended - Failed",
    "RA": "Audited",
    "AW": "Admin Withdrawn",
    "DR": "Admin Dropped",
    "B5": "Admin Dropped"
}

In [408]:
#Section 1f - Mapping the Codes to Translations
mtfull["Registration Status"] = mtfull["Registration Status"].map(regcodemap)
mtfull.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860


##### <center>*Creating a Major College Column*</center>

The Major College column lists what college the majors belong to. This will be useful in creating the advisor intervention contact sheet, as we can break students up by college so advisors can contact their unit's students easily.

In [409]:
#Section 1g - Major -> College Conditions & Outcome
majcond = [
    mtfull["Major"].isin(caedmajor),
    mtfull["Major"].isin(ccimajor),
    mtfull["Major"].isin(cotamajor),
    mtfull["Major"].isin(othermajor)] 

majout = ["CAED", "CCI", "CotA", "Other"]

In [410]:
#Section 1f - Plugging Conditions/Outcomes to Create Column
mtfull["Major College"] = np.select(majcond,majout, "Missing") #Use Missing Value to catch any missed majors
mtfull["Major College"].unique() #Run to check for missing majors - if no Missing, it worked

<StringArray>
['CAED', 'Other', 'CotA', 'CCI']
Length: 4, dtype: str

##### <center>*Creating a Subject College Column*</center>

Following the same logic as above, but with subjects. This helps assign courses to the college and allows for the creation of intervention tables later on (which are broken down by college).

In [411]:
#Section 1g - Subject -> College Conditions & Outcome
subcond = [
    mtfull["Subject"].isin(caedsubject),
    mtfull["Subject"].isin(ccisubject),
    mtfull["Subject"].isin(cotasubject)] 

subout = ["CAED", "CCI", "CotA"]

In [412]:
#Section 1h - Plugging Conditions/Outcomes to Create Column
mtfull["Subject College"] = np.select(subcond,subout, "Missing") #Use Missing Value to check for missed subjects
mtfull["Subject College"].unique() #Run to check for missing majors - if no Missing, it worked

<StringArray>
['CAED', 'CotA', 'CCI']
Length: 3, dtype: str

##### <center>*Assigning W/DR Grades*</center>

W grades are assigned to the final grade only when someone withdraws. Drops do not have a grade in the system, and leave NaN values. To account for this, W and DR grades are plugged in where appropriate to account for these situations.

In [413]:
#Section 1i - Withdrawl w/ no MT Check & W Grade Entry
mtfull["Withdrew No MT"] = np.where((mtfull["Final Grade"] == "W") & (mtfull["Mid Term Grade"].isna()), True, False) #checking for no W MT grade
mtfull["Mid Term Grade"] = np.where(mtfull["Withdrew No MT"] == True, "W", mtfull["Mid Term Grade"]) #Entering the W grades
mtfull["Mid Term Grade"].unique()

<StringArray>
[ 'B',  'A',  nan, 'A-',  'C', 'C+',  'D', 'B+', 'B-', 'C-',  'F',  'S', 'D+',
  'W',  'U', 'SF', 'NF']
Length: 17, dtype: str

In [414]:
#Section 1j - Dropped w/ no MT Check & DR Grade Entry
mtfull["Dropped no MT"] = np.where(((mtfull["Registration Status"] == "Std Dropped") | (mtfull["Registration Status"] == "Admin Dropped")) & (mtfull["Mid Term Grade"].isna()), True, False) #checking for Drop Status and no MT Grade
mtfull["Mid Term Grade"] = np.where(mtfull["Dropped no MT"] == True, "DR", mtfull["Mid Term Grade"]) #Entering the DR grades
mtfull["Mid Term Grade"].unique() 

<StringArray>
[ 'B',  'A',  nan, 'A-',  'C', 'C+',  'D', 'B+', 'B-', 'C-',  'F',  'S', 'D+',
 'DR',  'W',  'U', 'SF', 'NF']
Length: 18, dtype: str

In [415]:
#Section 1k - Dropped w/ no Final Check & DR Grade Entry
mtfull["Dropped no Fin"] = np.where(((mtfull["Registration Status"] == "Std Dropped") | (mtfull["Registration Status"] == "Admin Dropped")) & (mtfull["Final Grade"].isna()), True, False) #checking for dropped status and no MT grade
mtfull["Final Grade"] = np.where(mtfull["Dropped no Fin"] == True, "DR", mtfull["Final Grade"]) #Entering the DR grades
mtfull["Final Grade"].unique()

<StringArray>
[ 'A-',   'A',   'F',   'B',   nan,  'D+',   'C',  'B-',  'B+',  'C-',   'W',
  'C+',   'S',   'D',  'DR',  'XD',  'XF',  'SF',  'NF',  'AU', 'XSF', 'XC-',
  'IN', 'XD+',  'NR', 'XNF',   'U']
Length: 27, dtype: str

In [416]:
#Section 1l - Removing Check Columns
#These columns just add clutter and are not referenced again later, so removing them helps reading the dataframe easier
mtfull = mtfull.drop(["Dropped no Fin", "Dropped no MT", "Withdrew No MT"], axis=1)
mtfull.keys()

Index(['Record ID', 'Registration Status', 'Subject', 'Course', 'Campus',
       'Final Grade', 'Mid Term Grade', 'Department', 'Major',
       'Academic Period', 'Class', 'Course Code', 'Major College',
       'Subject College'],
      dtype='str')

##### <center>*Stripping X Final Grades*</center>

This isn't an issue with midterm grades, but when looking at historical grade data, some final grades have an X (due to records being cleared). This strips the X from the grade so it can be factored into analysis.

In [417]:
#Section 1m - Stripping X Grades
mtfull["Final Grade"] = mtfull["Final Grade"].str.lstrip("X")
mtfull["Final Grade"].unique()

<StringArray>
['A-',  'A',  'F',  'B',  nan, 'D+',  'C', 'B-', 'B+', 'C-',  'W', 'C+',  'S',
  'D', 'DR', 'SF', 'NF', 'AU', 'IN', 'NR',  'U']
Length: 21, dtype: str

##### <center>*Assiging Numerical Value to Midterm and Final Grades*</center>

The next step involves the creation of converting the grades to a numerical value. This conversion goes off of the university's GPA scale. There are some grades that do not correlate with a number due to not contributing toward GPA. Some of the grades assigned and logic can be found below:
- DR: will be an NaN. since they never earned a grade, there is no point in assigning a value since they would lead to a higher concentration of values in a certain spot.
- NR: will be an NaN. NR grades are assigned when the instructor never reported a grade.
- AU: will be an NaN - these are people auditing the course, so they never get a grade.
- S: will be a 4. S grades do not contribute to GPA, but they are a passing grade. Presumably there would be movement from S to U and vice versa, so U will be assigned a 0.
- U: will be a 0. See the logic for S
- W: will be a 0. W's do not contribute to GPA, but it is still a "negative" outcome for the course and is calculated in other retention efforts as a negative outcome (i.e. DFW rates are D, F and W grades).
- IN: will be an NaN. IN grades are put whenever students get an extension on entering their grades. It is interpreted similarly to W's (doesn't impact GPA), but since there is a chance for this to change, I don't want to count it (students could complete the work and come back with a passing grade)

*Grade Conversion Chart*
- A = 4.0
- A- = 3.7
- B+ = 3.3
- B = 3.0
- B- = 2.7
- C+ = 2.3
- C = 2.0
- C- = 1.7
- D+ = 1.3
- D = 1.0
- F = 0.0
- W = 0.0
- S = 4.0
- U = 0.0
- SF = 0.0
- NF =  0.0
- AU = NaN
- NaN = NaN
- DR =  NaN
- IN =  NaN
- NR = NaN

In [418]:
#Section 1n - Creating the GPA Map
gpamap = {"A":4.0, "A-":3.7, "B+":3.3, "B":3.0, "B-":2.7, "C+":2.3, "C":2.0, "C-":1.7, "D+":1.3,
          "D":1.0, "F":0.0, "W":0.0, "S":4.0, "U":0.0, "SF":0.0, "NF":0.0,
          "AU": "NaN", "DR":"NaN", "IN":"NaN", "NR":"NaN", "NaN":"NaN"}

In [419]:
#Section 1o - Applying the GPA Map and Create Number Columns
mtfull["Mid Term Grade Number"] = mtfull["Mid Term Grade"].map(gpamap)
mtfull["Final Grade Number"] = mtfull["Final Grade"].map(gpamap)
mtfull.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Mid Term Grade Number,Final Grade Number
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED,3.0,3.7
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860,CAED,CAED,4.0,4.0
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860,Other,CAED,4.0,4.0
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860,CAED,CAED,3.0,0.0
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED,3.0,3.0


##### <center>*Sorting Academic Periods from Oldest to Newest*</center>

Many of the later figures require the academic period column be sorted in chronological order. Doing this now helps set up future files for success (instead of having to run it many times later on).

In [420]:
#Section 1p - Sorting Academic Period 
mtfull = mtfull.sort_values("Academic Period")
mtfull["Academic Period"].unique()

array([202280, 202310, 202380, 202410, 202480, 202510, 202580, 202610])

##### <center>*Creating the All Students CSV File*</center>

The csv file created here will be used for the following figures:
- Midterm Grade Summary Table - A look at how many students passed, failed, or had no midterm grade reported.
- Midterm Grade Summary Over Time - A look at how midterm grades fluctuated over time.

Further data manipulation of this file/frame can be found in [Section 2](#section-2---midterm-grade-summary-tables).

In [421]:
#Section 1q - Creating the Full Data File
mtfull.to_csv("mtfull.csv")

##### <center>*Removal of Non-Hub Majors*</center>

In [422]:
#Section 1r - Removal of Non-Hub Majors
mthubonly = mtfull[mtfull["Major"] != "OTH"].copy()
mthubonly.shape

(58787, 16)

##### <center>*Advisor Intervention Columns*</center>

The advising office has laid out criteria for what grades require interventions from advisors. There was a switch for the freshman/sophomore criteria in Spring 2026, so a separate variable is needed for that. In the future, this could be removed when we are only looking at Spring 2026 grades onwards.

In [423]:
#Section 1s - Intervention Conditions/Outcomes
intercond = [
    ((mthubonly["Mid Term Grade"].isin(pre202610frsopassgrade)) & (mthubonly["Class"].isin(frsocheck)) & (mthubonly["Academic Period"].isin(pre202610terms))),
    ((mthubonly["Mid Term Grade"].isin(pre202610frsofailgrade)) & (mthubonly["Class"].isin(frsocheck)) & (mthubonly["Academic Period"].isin(pre202610terms))),
    ((mthubonly["Mid Term Grade"].isin(post202610frsopassgrade)) & (mthubonly["Class"].isin(frsocheck))  & (mthubonly["Academic Period"].isin(post202610terms))),
    ((mthubonly["Mid Term Grade"].isin(post202610frsofailgrade)) & (mthubonly["Class"].isin(frsocheck))  & (mthubonly["Academic Period"].isin(post202610terms))),
    ((mthubonly["Mid Term Grade"].isin(jrsrpassgrade)) & (mthubonly["Class"].isin(jrsrcheck))),
    ((mthubonly["Mid Term Grade"].isin(jrsrfailgrade)) & (mthubonly["Class"].isin(jrsrcheck))),
    mthubonly["Mid Term Grade"].isin(dropgrade),
    mthubonly["Mid Term Grade"].isin(wgrade),
    mthubonly["Mid Term Grade"].isin(nfgrade)
]

interout = ["Passing MT Grade", "FR/SO Intervention Needed", "Passing MT Grade", "FR/SO Intervention Needed", "Passing MT Grade", "JR/SR Intervention Needed", "Dropped Course", "Withdrew From Course", "Never Attended Course"]

In [424]:
#Section 1t - Applying Intervention Conditions/Outcomes
mthubonly["MT Intervention Needed"] = np.select(intercond,interout, "No MT Grade Entered") #This will catch any students who do not meet any of the criteria
mthubonly.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Mid Term Grade Number,Final Grade Number,MT Intervention Needed
40,40,Registered,AED,22860,KC,B,NaN,ARCH,ID,202280,JR,AED 22860,CAED,CAED,NaN,3.0,No MT Grade Entered
39,39,Registered,AED,22860,KC,C-,C+,ARCH,ID,202280,FR,AED 22860,CAED,CAED,2.3,1.7,Passing MT Grade
38,38,Registered,AED,22860,KC,B-,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED,3.0,2.7,Passing MT Grade
37,37,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860,CAED,CAED,4.0,4.0,Passing MT Grade
36,36,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860,CAED,CAED,4.0,4.0,Passing MT Grade


##### <center>*Creating the Hub Only CSV File*</center>

The csv file created here will be used for the following figures:
- MT -> Final Grade Heatmap - A look at how midterm and final grades changed for students who received interventions from advisors.

Further data manipulation of this file/frame can be found in Section

In [425]:
#Section 1u - Creating the Hub Only CSV File
mthubonly.to_csv("mthubonlyclean.csv")

##### <center>*Removing Non-Intervention Students*</center>

In [426]:
#Section 1v - Removing Non-Intervention Students
mtinterventions = mthubonly[((mthubonly["MT Intervention Needed"] == "FR/SO Intervention Needed") | (mthubonly["MT Intervention Needed"] == "JR/SR Intervention Needed"))].copy()
mtinterventions["MT Intervention Needed"].unique()

<StringArray>
['FR/SO Intervention Needed', 'JR/SR Intervention Needed']
Length: 2, dtype: str

##### <center>*Creating the Intervention Only CSV File*</center>

The csv file created here will be used for the following figures:
- Intervention Contact List - An Excel file that will provide the contact information for students who require an intervention based on midterm grades.
- Interventions Over Time Line Chart - A line chart that shows how many interventions were needed over time.

Further data manipulation of this file/frame can be found in Section 

In [427]:
#Section 1w - Creating the Intervention Only CSV File
mtinterventions.to_csv("mtinterventions.csv")

#### <center>**Section 2 - Midterm Grade Summary Tables**</center>

The code in this section is focused on creating multiple tables that summarize the Midterm grade data by college. The purpose of this table is to provide college leadership (Dean and School Directors) an overview of how well students are doing on midterms. Through this, leadership can understand pain points for students, perform outreach to instructors and guide discussions for curricular revisions.

Note that this table looks at **all** enrolled students - focus is not placed on hub-only majors, as a majority of these students can be taken by any student. As such, it will be referencing the **mtfull.csv** file.

Tables will allow filtering by either department (CCI and CotA) or subject (CAED). This is due to multiple subjects being within CCI and CotA departments, while CAED does not have individual departments. 

##### <center>*Reloading Data*</center>

In [428]:
#Section 2a - Reloading the Data
mtsum = pd.read_csv("mtfull.csv")

##### <center>*Removing Dropped Students*</center>

As dropped students do not have a formal grade (and are not regularly looked at when considering things like DFW rates), they are removed from the calculation.

In [429]:
#Section 2b - Removing Dropped Students
mtsum = mtsum[mtsum["Mid Term Grade"] != "DR"]
mtsum["Mid Term Grade"].unique()

<StringArray>
[ nan, 'C+',  'B',  'A', 'C-', 'A-',  'D',  'C',  'F', 'B+',  'S', 'B-',  'W',
 'D+', 'SF',  'U', 'NF']
Length: 17, dtype: str

##### <center>*Defining Pass/Fail Conditions*</center>

Separate from the interventions, different criteria are laid out to determine if a student is passing/failing. This falls in line with standard DFW rates, with the exception of C- being considered a part of the DFW end. I kept this to keep in line with past reporting standards, but also because some programs require higher grades than C- to count credit toward completion.

In [430]:
#Section 2c - Pass/Fail Conditions & Outcomes
mtstatus = [
    mtsum["Mid Term Grade"].isin(mtpass),
    mtsum["Mid Term Grade"].isin(mtfail)
]

mtout = ["MT C or Higher","MT C-, D, F, W"]

In [431]:
#Section 2d - Applying Pass/Fail Conditions & Outcomes
mtsum["MT Status"] = np.select(mtstatus, mtout, "MT Not Reported")
mtsum["MT Status"].unique()

<StringArray>
['MT Not Reported', 'MT C or Higher', 'MT C-, D, F, W']
Length: 3, dtype: str

##### <center>*Creating Summary Table*</center>

In order to capture the totals for each course, the data is pivoted to summarize the different rates. This helps not only with the creation of the tables, but also in the line chart later on.

Something that could be considered in future iterations is rerunning some of the above code with college majors only (i.e. only people in CCI when looking at CCI courses). This can give an idea of how students in the college are doing in these core classes as a whole, as opposed to all students (which includes non-majors who maybe are not as interested in the material/clearing it for a gen ed).

In [432]:
#Section 2e - Creating the Summary Table
mtsum = mtsum.value_counts(["Course Code", "Course", "Academic Period","Subject","Subject College","Department","MT Status"]).reset_index() #Converts the data into a pivot table that counts how many instances exist of each key combo
mtsum = mtsum.pivot(index=["Course Code", "Course", "Academic Period","Subject","Subject College","Department"], #sets the indices for the table
                            columns = "MT Status",
                            values = "count",).fillna(0).reset_index().rename_axis(None, axis=1)
                            #In order above - filling empty values with 0, resetting hte index so it is flat, and then removing MT Status as a column with counts

mtsum.head()

,Course Code,Course,Academic Period,Subject,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported
0,AED 22860,22860,202280,AED,CAED,ARCH,84.0,6.0,13.0
1,AED 22860,22860,202310,AED,CAED,ARCH,8.0,5.0,1.0
2,AED 22860,22860,202380,AED,CAED,ARCH,58.0,12.0,9.0
3,AED 22860,22860,202410,AED,CAED,ARCH,10.0,1.0,1.0
4,AED 22860,22860,202480,AED,CAED,ARCH,75.0,11.0,7.0


##### <center>*Creating a Total Grades Column*</center>

In [433]:
#Section 2f - Creating a Total Grades Column
mtsum["Total Grades"] = mtsum["MT C or Higher"] + mtsum["MT C-, D, F, W"] + mtsum["MT Not Reported"]
mtsum.head()

,Course Code,Course,Academic Period,Subject,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades
0,AED 22860,22860,202280,AED,CAED,ARCH,84.0,6.0,13.0,103.0
1,AED 22860,22860,202310,AED,CAED,ARCH,8.0,5.0,1.0,14.0
2,AED 22860,22860,202380,AED,CAED,ARCH,58.0,12.0,9.0,79.0
3,AED 22860,22860,202410,AED,CAED,ARCH,10.0,1.0,1.0,12.0
4,AED 22860,22860,202480,AED,CAED,ARCH,75.0,11.0,7.0,93.0


##### <center>*Creating a Term Total Observation*</center>

Something that I'd like to add is how well the subject did as a whole in a term. The logic behind it was highlighting how an entire subject did during a term and how grades fluctuate between different academic periods for the subjects. This could highlight ebbs and flows in grade data, and patterns of subjects that experience major issues.

In [434]:
#Section 2g - Creating the Term Total Observations
mtsumdept = mtsum.groupby(["Subject", "Academic Period", "Subject College", "Department"])[gradecols].sum().reset_index() #Makes a version of the table that has term totals (the total numbers of grades assigned by subject each term)
mtsumdept["Course Code"] = mtsumdept["Subject"] + " - Term Total" #Creates a Course Code for it called "Subject - Term Total"
mtsumdept = mtsumdept[["Course Code", "Academic Period", "Subject", "Subject College", "Department", "MT C or Higher", "MT C-, D, F, W", "MT Not Reported", "Total Grades"]] #Reordering to align with the original table
mtsumdept.head() #running to make sure it worked

,Course Code,Academic Period,Subject,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades
0,AED - Term Total,202280,AED,CAED,ARCH,84.0,6.0,13.0,103.0
1,AED - Term Total,202310,AED,CAED,ARCH,8.0,5.0,1.0,14.0
2,AED - Term Total,202380,AED,CAED,ARCH,58.0,12.0,9.0,79.0
3,AED - Term Total,202410,AED,CAED,ARCH,10.0,1.0,1.0,12.0
4,AED - Term Total,202480,AED,CAED,ARCH,75.0,11.0,7.0,93.0


In [435]:
#Section 2h - Combining the Frames and Resorting the Term
mtsum = pd.concat([mtsum, mtsumdept])
mtsum = mtsum.sort_values("Academic Period")
mtsum

,Course Code,Course,Academic Period,Subject,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades
0,AED 22860,22860.0,202280,AED,CAED,ARCH,84.0,6.0,13.0,103.0
127,VCD - Term Total,NaN,202280,VCD,CCI,VCD,627.0,97.0,76.0,800.0
119,THEA - Term Total,NaN,202280,THEA,CotA,THDN,694.0,87.0,82.0,863.0
44,ARCH 22860,22860.0,202280,ARCH,CAED,ARCH,84.0,18.0,6.0,108.0
52,ARCH 24669,24669.0,202280,ARCH,CAED,ARCH,72.0,17.0,13.0,102.0
...,...,...,...,...,...,...,...,...,...,...
118,MUS - Term Total,NaN,202610,MUS,CotA,MUS,877.0,150.0,82.0,1109.0
110,MDJ - Term Total,NaN,202610,MDJ,CCI,MDJ,765.0,113.0,88.0,966.0
27,ARCH 15217,15217.0,202610,ARCH,CAED,ARCH,117.0,18.0,17.0,152.0
23,ARCH 14841,14841.0,202610,ARCH,CAED,ARCH,74.0,18.0,9.0,101.0


##### <center>*Creating Pass/Fail/No Report Percents*</center>

Originally, I thought about just counting the total numbers for each subject and using that for future line charts. However, this would create an issue where some courses would be thrown way off because of semesters where there are more students enrolled due to more sections/increasing section counts. This could make the chart look uneven due to major differences in number. To make it more even, I opted to do percentages instead - this way we are looking out of 100 as opposed to varying numbers. The specific counts can be plugged into the hovertext for each term.

In [436]:
#Section 2i - Creating the Decimal #
mtsum["# of Passing Grades"] = mtsum["MT C or Higher"]/mtsum["Total Grades"]
mtsum["# of Failing Grades"] = mtsum["MT C-, D, F, W"]/mtsum["Total Grades"]
mtsum["# of Unreported Grades"] = mtsum["MT Not Reported"]/mtsum["Total Grades"]
mtsum.head()

,Course Code,Course,Academic Period,Subject,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,# of Failing Grades,# of Unreported Grades
0,AED 22860,22860.0,202280,AED,CAED,ARCH,84.0,6.0,13.0,103.0,0.815534,0.058252,0.126214
127,VCD - Term Total,NaN,202280,VCD,CCI,VCD,627.0,97.0,76.0,800.0,0.783750,0.121250,0.095000
119,THEA - Term Total,NaN,202280,THEA,CotA,THDN,694.0,87.0,82.0,863.0,0.804171,0.100811,0.095017
44,ARCH 22860,22860.0,202280,ARCH,CAED,ARCH,84.0,18.0,6.0,108.0,0.777778,0.166667,0.055556
52,ARCH 24669,24669.0,202280,ARCH,CAED,ARCH,72.0,17.0,13.0,102.0,0.705882,0.166667,0.127451


In [437]:
#Section 2j - Converting to %
mtsum["% of Passing Grades"] = mtsum["# of Passing Grades"].apply("{:.2%}".format)
mtsum["% of Failing Grades"] = mtsum["# of Failing Grades"].apply("{:.2%}".format)
mtsum["% of Unreported Grades"] = mtsum["# of Unreported Grades"].apply("{:.2%}".format)
mtsum.head()

,Course Code,Course,Academic Period,Subject,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,# of Failing Grades,# of Unreported Grades,% of Passing Grades,% of Failing Grades,% of Unreported Grades
0,AED 22860,22860.0,202280,AED,CAED,ARCH,84.0,6.0,13.0,103.0,0.815534,0.058252,0.126214,81.55%,5.83%,12.62%
127,VCD - Term Total,NaN,202280,VCD,CCI,VCD,627.0,97.0,76.0,800.0,0.783750,0.121250,0.095000,78.38%,12.12%,9.50%
119,THEA - Term Total,NaN,202280,THEA,CotA,THDN,694.0,87.0,82.0,863.0,0.804171,0.100811,0.095017,80.42%,10.08%,9.50%
44,ARCH 22860,22860.0,202280,ARCH,CAED,ARCH,84.0,18.0,6.0,108.0,0.777778,0.166667,0.055556,77.78%,16.67%,5.56%
52,ARCH 24669,24669.0,202280,ARCH,CAED,ARCH,72.0,17.0,13.0,102.0,0.705882,0.166667,0.127451,70.59%,16.67%,12.75%


In [438]:
#Section 2k - Converting to Number W/O %
#Some future measurements play nicer with round numbers as opposed to percents, so this helps remedy that.
mtsum["# of Passing Grades"] = (mtsum["# of Passing Grades"] * 100).round(2)
mtsum["# of Failing Grades"] = (mtsum["# of Failing Grades"] * 100).round(2)
mtsum["# of Unreported Grades"] = (mtsum["# of Unreported Grades"] * 100).round(2)
mtsum.head()

,Course Code,Course,Academic Period,Subject,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,# of Failing Grades,# of Unreported Grades,% of Passing Grades,% of Failing Grades,% of Unreported Grades
0,AED 22860,22860.0,202280,AED,CAED,ARCH,84.0,6.0,13.0,103.0,81.55,5.83,12.62,81.55%,5.83%,12.62%
127,VCD - Term Total,NaN,202280,VCD,CCI,VCD,627.0,97.0,76.0,800.0,78.38,12.12,9.50,78.38%,12.12%,9.50%
119,THEA - Term Total,NaN,202280,THEA,CotA,THDN,694.0,87.0,82.0,863.0,80.42,10.08,9.50,80.42%,10.08%,9.50%
44,ARCH 22860,22860.0,202280,ARCH,CAED,ARCH,84.0,18.0,6.0,108.0,77.78,16.67,5.56,77.78%,16.67%,5.56%
52,ARCH 24669,24669.0,202280,ARCH,CAED,ARCH,72.0,17.0,13.0,102.0,70.59,16.67,12.75,70.59%,16.67%,12.75%


In [439]:
#Section 2l - Sorting # of Passing Grades to Have Priority
mtsum = mtsum.sort_values("# of Passing Grades", ascending = False)

##### <center>*Splitting the Frame Into Two*</center>

In order to create the table and line chart, we are going to make two versions of the dataframe - one with term totals (that will be used for the line chart) and another w/o (to be used for the table).

In [440]:
#Section 2m - Splitting the Frame
mtsumline = mtsum #Version to be used for the line chart later - just takes the current version
mtsumtab = mtsum[~mtsum["Course Code"].str.contains("Term Total", na=False)].copy() #Version to be used for the table - this drops any observation with Term Total

##### <center>*Defining Hub-Specific Dataframes*</center>

This will break the figures into their own dataframes, as well as assigning the color patterns defined before (both biological relevance and alternating color rows)

In [441]:
#Section 2l - Creating Unique Summary Tables
caedsumtab = mtsumtab[(mtsumtab["Subject College"] == "CAED") & (mtsumtab["Academic Period"].isin(currentterm))] #Makes the unique table for the college
caedsub = sorted(caedsumtab["Subject"].unique()) #Creates the sorted list of subjects for the dropdown
ccisumtab = mtsumtab[(mtsumtab["Subject College"] == "CCI") & (mtsumtab["Academic Period"].isin(currentterm))]
ccisch = sorted(ccisumtab["Department"].unique())
cotasumtab = mtsumtab[(mtsumtab["Subject College"] == "CotA") & (mtsumtab["Academic Period"].isin(currentterm))]
cotasch = sorted(cotasumtab["Department"].unique())

In [442]:
#Section 2m - Making College-Specific Pass Rate Colors
caedpass = []
for num in caedsumtab["# of Passing Grades"]:
    if num >= 85:
        caedpass.append("#CCFFCC")
    elif num >= 80:
        caedpass.append("#FFFFCC")
    else:
        caedpass.append("#FFCCCC")

ccipass = []
for num in ccisumtab["# of Passing Grades"]:
    if num >= 85:
        ccipass.append("#CCFFCC")
    elif num >= 80:
        ccipass.append("#FFFFCC")
    else:
        ccipass.append("#FFCCCC")

cotapass = []
for num in cotasumtab["# of Passing Grades"]:
    if num >= 85:
        cotapass.append("#CCFFCC")
    elif num >= 80:
        cotapass.append("#FFFFCC")
    else:
        cotapass.append("#FFCCCC")

In [443]:
#Section 2n - Creating Zebra Rows (Alternating Colors)
caedrows = len(caedsumtab)
caedzebra = ["#FFFFFF", "#FFF1CC"] * (caedrows // 2+1)
caedzebra = caedzebra[:caedrows]

ccirows = len(ccisumtab)
ccizebra = ["#FFFFFF", "#FFF1CC"] * (ccirows // 2+1)
ccizebra = ccizebra[:ccirows]

cotarows = len(cotasumtab)
cotazebra = ["#FFFFFF", "#FFF1CC"] * (cotarows // 2+1)
cotazebra = cotazebra[:cotarows]

##### <center>*Creating the Summary Tables*</center>

This table is based off of the table created by my predecessor in Excel. It is sorted to highlight how many students are in each group, the total number of students, and the percent who passed.

In [444]:
#Section 2o - CAED's Summary Table
caedtraces = [] #creating the empty list for the traces
caedbuttons = [] #creating the empty list for the buttons

#Creates the "All" State (Default)
caedtraces.append(go.Table(
    header = dict(values = headers, #In order to make the headers stand out, they are given the gold color with bold text
                  fill_color = "#EFAB00",
                  line_color = "black",
                  font_color = "black",
                  font_weight = "bold",
                  font_size = 12),
    cells = dict(values = [caedsumtab[v] for v in values], #pulls the different values for the "all" state (i.e. the pass rate, number of students and so on)
                 fill_color = [caedzebra, caedzebra, caedzebra, caedzebra, caedzebra, caedpass], #assigns the alternating colors, followed by whether the course was passed/failed
                 line_color = "black",
                 font_size = 12),
    visible = True                  
))

caedbuttons.append(dict( #setting the criteria for the "default" button
    label = "All",
    method = "update",
    args = [{"visible":[True] + [False] * len(caedsub #sets the all table as visible, and the rest as hidden (hence the length of the subjects)
    )}]
))

#Creates states for each subject
for i, caedsub in enumerate(caedsubject): #goes through all the subjects in the frame
    caedsubjects = caedsumtab[caedsumtab["Subject"] == caedsub].copy() #creates a copy of the frame for the subject
    caedsubjects = caedsubjects.sort_values(by = "% of Passing Grades", ascending = False) #this helps sort the tables so the highest passing grades are first, followed by lower
    caedsubrows = len(caedsubjects) 
    caedsubpass = [] #since we are regenerating the frame, we have to rereun the passing grade logic here

    for num in caedsubjects["# of Passing Grades"]:
        if num >= 85:
            caedsubpass.append("#CCFFCC")
        elif num >= 80:
            caedsubpass.append("#FFFFCC")
        else:
            caedsubpass.append("#FFCCCC")

    caedtraces.append(go.Table(# this takes the same logic above, just applying it to the subframes
        header = dict(values = headers, 
                  line_color = "black",
                  fill_color = "#EFAB00",
                  font_color = "black",
                  font_weight = "bold",
                  font_size = 12),
        cells = dict(values = [caedsubjects[v] for v in values],
                 fill_color = [caedzebra, caedzebra, caedzebra, caedzebra, caedzebra, caedsubpass],
                 line_color = "black"),
        visible = False          
    ))

    caedsubvisibility = [False] * (len(caedsubject) + 1) #This is setting all the other figures to hidden
    caedsubvisibility[i + 1] = True #And this sets whatever the "current" figure is to be shown

    caedbuttons.append(dict(
        label = caedsub,
        method = "update",
        args = [{"visible":caedsubvisibility}]
    ))

caedsumlayout = go.Layout(
    title=dict(text= f"<b>{prettyterm} CAED Midterm Report", #prettyterm is defined earlier - this ensures the table always specifies the right term
                font_weight = "bold", 
                font_color = "black", 
                xanchor = "center", 
                yanchor = "top", 
                x = .5,
                y = .965,
                font_size = 24),
    height = 650,
    width = 1200
)

caedsumtable = go.Figure(data = caedtraces, layout = caedsumlayout)

caedsumtable.update_layout(updatemenus = [dict(
    type = "dropdown", #I like the dropdown menus more - they are less intrusive in my opinion, and oftentimes do not get so long it is a hassle to scroll through
    direction = "down",
    showactive = True,
    xanchor = "center",
    yanchor = "top",
    x = .5,
    y = 1.1,
    buttons = caedbuttons)],
    annotations=[dict( #In order to specify what the dropdown was, I added an annotation indicating it is the subject.
        text = "<b>Subject:</b>", 
        showarrow = False,
        xanchor = "center",
        yanchor = "top",
        x = .435,
        y = 1.0875,
        font_size = 14,
        font_color = "black"
    )],
    font_family = "Segoe UI",
    )

caedsumtable.show()

In [445]:
#Section 2p - CCI Table
ccitraces = []
ccibuttons = []

ccitraces.append(go.Table(
    header = dict(values = headers,
                  line_color = "black",
                  fill_color = "#EFAB00",
                  font_color = "black",
                  font_weight = "bold",
                  font_size = 12),
    cells = dict(values = [ccisumtab[v] for v in values],
                 fill_color = [ccizebra, ccizebra, ccizebra, ccizebra, ccizebra, ccipass],
                 line_color = "black",
                 font_size = 12),
    visible = True                  
))

ccibuttons.append(dict(
    label = "All",
    method = "update",
    args = [{"visible":[True] + [False] * len(ccisch)}]
))

for i, ccischool in enumerate(ccisch):
    ccischools = ccisumtab[ccisumtab["Department"] == ccischool].copy()
    ccischools = ccischools.sort_values(by = "% of Passing Grades", ascending = False)
    ccischrows = len(ccischools)
    ccischpass = []

    for num in ccischools["# of Passing Grades"]:
        if num >= 85:    
            ccischpass.append("#CCFFCC")
        elif num >= 80:
            ccischpass.append("#FFFFCC")
        else:
            ccischpass.append("#FFCCCC")

    ccitraces.append(go.Table(
        header = dict(values = headers,
                  line_color = "black",
                  fill_color = "#EFAB00",
                  font_color = "black",
                  font_weight = "bold",
                  font_size = 12),
        cells = dict(values = [ccischools[v] for v in values],
                 fill_color = [ccizebra, ccizebra, ccizebra, ccizebra, ccizebra, ccischpass],
                 line_color = "black"),
        visible = False          
    ))

    ccischvisibility = [False] * (len(ccisch) + 1)
    ccischvisibility[i + 1] = True

    ccibuttons.append(dict(
        label = ccischool,
        method = "update",
        args = [{"visible":ccischvisibility}]
    ))

ccisumlayout = go.Layout(
    title=dict(text= f"<b>{prettyterm} CCI Midterm Report",
                font_weight = "bold", 
                font_color = "black", 
                xanchor = "center", 
                yanchor = "top", 
                x = .5,
                y = .965,
                font_size = 24),
    height = 650,
    width = 1200
)

ccisumtable = go.Figure(data = ccitraces, layout = ccisumlayout)

ccisumtable.update_layout(updatemenus = [dict(
    type = "dropdown",
    direction = "down",
    showactive = True,
    xanchor = "center",
    yanchor = "top",
    x = .5,
    y = 1.1,
    buttons = ccibuttons)],
    annotations=[dict( #As CCI and CotA follow a school structure, I opted to make this one the school - that way directors can view all the subjects in their unit.
        text = "<b>School:</b>", 
        showarrow = False,
        xanchor = "center",
        yanchor = "top",
        x = .435,
        y = 1.0875,
        font_size = 14,
        font_color = "black"
    )],
    font_family = "Segoe UI")

ccisumtable.show()

In [446]:
#Section 2p - CotA Table
cotatraces = []
cotabuttons = []

cotatraces.append(go.Table(
    header = dict(values = headers,
                  line_color = "black",
                  fill_color = "#EFAB00",
                  font_color = "white",
                  font_weight = "bold",
                  font_size = 12),
    cells = dict(values = [cotasumtab[v] for v in values],
                 fill_color = [cotazebra, cotazebra, cotazebra, cotazebra, cotazebra, cotapass],
                 line_color = "black",
                 font_size = 12),
    visible = True                  
))

cotabuttons.append(dict(
    label = "All",
    method = "update",
    args = [{"visible":[True] + [False] * len(cotasch)}]
))

for i, cotaschool in enumerate(cotasch):
    cotaschools = cotasumtab[cotasumtab["Department"] == cotaschool].copy()
    cotaschools = cotaschools.sort_values(by = "% of Passing Grades", ascending = False)
    cotaschrows = len(cotaschools)
    cotaschpass = []

    for num in cotaschools["# of Passing Grades"]:
        if num >= 85:    
            cotaschpass.append("#CCFFCC")
        elif num >= 80:
            cotaschpass.append("#FFFFCC")
        else:
            cotaschpass.append("#FFCCCC")

    cotatraces.append(go.Table(
        header = dict(values = headers,
                  line_color = "black",
                  fill_color = "#EFAB00",
                  font_color = "white",
                  font_weight = "bold",
                  font_size = 12),
        cells = dict(values = [cotaschools[v] for v in values],
                 fill_color = [cotazebra, cotazebra, cotazebra, cotazebra, cotazebra, cotaschpass],
                 line_color = "black"),
        visible = False          
    ))

    cotaschvisibility = [False] * (len(cotasch) + 1)
    cotaschvisibility[i + 1] = True

    cotabuttons.append(dict(
        label = cotaschool,
        method = "update",
        args = [{"visible":cotaschvisibility}]
    ))

cotasumlayout = go.Layout(
    title=dict(text= f"<b>{prettyterm} CotA Midterm Report",
                font_weight = "bold", 
                font_color = "black", 
                xanchor = "center", 
                yanchor = "top", 
                x = .5,
                y = .9675,
                font_size = 24),
    height = 650,
    width = 1200
)

cotasumtable = go.Figure(data = cotatraces, layout = cotasumlayout)

cotasumtable.update_layout(updatemenus = [dict(
    type = "dropdown",
    direction = "down",
    showactive = True,
    xanchor = "center",
    yanchor = "top",
    x = .5,
    y = 1.1,
    buttons = cotabuttons)],
    annotations=[dict(
        text = "<b>School:</b>", 
        showarrow = False,
        xanchor = "center",
        yanchor = "top",
        x = .435,
        y = 1.0875,
        font_size = 14,
        font_color = "black"
    )],
    font_family = "Segoe UI")

cotasumtable.show()

#### <center>**Section 3 - Midterm Grade Summary Over Time Line Charts**</center>

The code in this section is focused on creating line charts for the units that show how their courses have done over time. This is done via a line chart that plots the three different rates (Pass, Fail, No Report) to show changes over time.

Once again, this table will look at the changes over time across **all** students. However, instead of pulling the CSV file, we are calling the previously made **mtsumline** variable, as it has already been prepared to be plugged into the figure.

##### <center>*Creating Subject/Department Dataframes*</center>

As touched on before - CAED will look at subjects, while CCI and CotA will look at subjects.

In [447]:
#Section 3a - Resorting the Academic Period Column
mtsumline = mtsumline.sort_values("Academic Period")
mtsumline["Academic Period"].unique()

array([202280, 202310, 202380, 202410, 202480, 202510, 202580, 202610])

In [448]:
mtsumline["Academic Period"] = mtsumline["Academic Period"].map(figtermmap)
mtsumline["Academic Period"].unique()

<StringArray>
[  'Fall 2022', 'Spring 2023',   'Fall 2023', 'Spring 2024',   'Fall 2024',
 'Spring 2025',   'Fall 2025', 'Spring 2026']
Length: 8, dtype: str

In [449]:
#Section 3b - Creating Unit Specific Dataframes
aedmtline = mtsumline[mtsumline["Subject"] == "AED"]
archmtline = mtsumline[mtsumline["Subject"] == "ARCH"]
arcsmtline = mtsumline[mtsumline["Subject"] == "ARCS"]
cmgtmtline = mtsumline[mtsumline["Subject"] == "CMGT"]
idmtline = mtsumline[mtsumline["Subject"] == "ID"]
ccimtline = mtsumline[mtsumline["Department"] == "CCI"]
commmtline = mtsumline[mtsumline["Department"] == "COMM"]
ematmtline = mtsumline[mtsumline["Department"] == "EMAT"]
mdjmtline = mtsumline[mtsumline["Department"] == "MDJ"]
vcdmtline = mtsumline[mtsumline["Department"] == "VCD"]
artmtline = mtsumline[mtsumline["Department"] == "ART"]
fdmmtline = mtsumline[mtsumline["Department"] == "FDM"]
musmtline = mtsumline[mtsumline["Department"] == "MUS"]
thdnmtline = mtsumline[mtsumline["Department"] == "THDN"]

##### <center>*Creating Line Charts*</center>

When translating figures to PowerBI, the HTML string generated may exceed the character limit that PowerBI can read (32,767). In order to counteract this, select figures are broken into smaller chunks to control the size of the HTML string so it is readable. The following units are impacted by this: ART, FDM, MDJ, MUS, THDN and VCD. 

*The high score for character count was the full THDN chart - 90,000+ characters*

In [450]:
#Section 3c - AED Line Chart

#New sorting method - without this, the dropdowns are not organized properly
term_total = ["AED - Term Total"] #pulls term total
other_courses = (
    aedmtline[aedmtline["Course Code"] != "AED - Term Total"] #removes term total
    .drop_duplicates(subset=["Course Code"]) #removes duplicate entries (i.e. lists AED 1 once and removes other instances)
    .sort_values("Course")["Course Code"] #Sorts the Course Code (Text & Number combo) values by the Course value (the number)
    .tolist() #converts it to a list
)
courses = term_total + other_courses #combines the list into one big ordered list

coursebuttons = []

aedratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = aedmtline[aedmtline["Course Code"] == course] 
    is_visible = (i == 0)

    aedratelines.add_trace(go.Scatter( #creates the pass rate line
        x = coursepct["Academic Period"].tolist(), #this is necessary to get the HTML to read all the traces (the JSON file doesn't read it otherwise)
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2), #Green since it is passing - biological relevance
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), #I wanted to include how many specific students met the criteria somehow - this gets it in hte hovertemplate
        hovertemplate = ( #This is what will give the more specific information (how many people passed it)
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    aedratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2), #red since it is failing - biological relevance
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    aedratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2), #Gray since it is neutral - biological relevance
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    aedratelines.add_trace(go.Scatter( #I wanted to provide some reference of how many were enrolled in each term - this provided a non-intrusive way of doing so, and provides context of how many are in a term (a failing term where there are a lot of students or a passing term where there aren't many can be outliers)
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"), #to help it stand out
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]), #Fixing the placement helps keep it out of the way, plus sets one every period
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4) #Since there are four traces, this hides all of the figures and traces
    visibility[i*4: i*4 + 4] = [True, True, True, True] #This sets the "active" one to show all of the traces

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    aedratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105], #I set the range to be bigger/smaller to allow the numbers and 100%'s stand out more
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>AED Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict( #Same logic as before - helps give an idea of what people are looking at
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict( #Same as above - helps people know what the dropdown is for
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .385,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict( #This helps the numbers stand out - without it, they are sitting over the grid and it looks not that great.
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

aedratelines.show()

In [451]:
#Section 3cd - ARCH Line Chart

term_total = ["ARCH - Term Total"] 
other_courses = (
    archmtline[archmtline["Course Code"] != "ARCH - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = term_total + other_courses

coursebuttons = []

archratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = archmtline[archmtline["Course Code"] == course] 
    is_visible = (i == 0)

    archratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    archratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    archratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    archratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    archratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>ARCH Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .385,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

archratelines.show()

In [452]:
#Section 3e - ARCS Line Chart
term_total = ["ARCS - Term Total"] 
other_courses = (
    arcsmtline[arcsmtline["Course Code"] != "ARCS - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = term_total + other_courses
coursebuttons = []

arcsratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = arcsmtline[arcsmtline["Course Code"] == course] 
    is_visible = (i == 0)

    arcsratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    arcsratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    arcsratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    arcsratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    arcsratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>ARCS Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .385,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

arcsratelines.show()

In [453]:
#Section 3f - CMGT Line Chart
term_total = ["CMGT - Term Total"] 
other_courses = (
    cmgtmtline[cmgtmtline["Course Code"] != "CMGT - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = term_total + other_courses
coursebuttons = []

cmgtratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = cmgtmtline[cmgtmtline["Course Code"] == course] 
    is_visible = (i == 0)

    cmgtratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    cmgtratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    cmgtratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    cmgtratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    cmgtratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>CMGT Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .385,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

cmgtratelines.show()

In [454]:
#Section 3g - ID Line Chart
term_total = ["ID - Term Total"] 
other_courses = (
    idmtline[idmtline["Course Code"] != "ID - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = term_total + other_courses
coursebuttons = []

idratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = idmtline[idmtline["Course Code"] == course] 
    is_visible = (i == 0)

    idratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    idratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    idratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    idratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    idratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>ID Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .385,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

idratelines.show()

In [455]:
#Section 3h - CCI Line Chart
term_total = ["CCI - Term Total"] 
other_courses = (
    ccimtline[ccimtline["Course Code"] != "CCI - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = term_total + other_courses
coursebuttons = []

cciratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = ccimtline[ccimtline["Course Code"] == course] 
    is_visible = (i == 0)

    cciratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    cciratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    cciratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    cciratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    cciratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>CCI Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .385,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

cciratelines.show()

In [456]:
#Section 3i - COMM Line Chart
term_total = ["COMM - Term Total"] 
other_courses = (
    commmtline[commmtline["Course Code"] != "COMM - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = term_total + other_courses
coursebuttons = []

commratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = commmtline[commmtline["Course Code"] == course] 
    is_visible = (i == 0)

    commratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    commratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    commratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    commratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    commratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>COMM Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .38,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

commratelines.show()

In [457]:
#Section 3j - EMAT Line Chart
term_total = ["EMAT - Term Total"] 
other_courses = (
    ematmtline[ematmtline["Course Code"] != "EMAT - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = term_total + other_courses
coursebuttons = []

ematratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = ematmtline[ematmtline["Course Code"] == course] 
    is_visible = (i == 0)

    ematratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ematratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ematratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    ematratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ematratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>EMAT Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .38,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

ematratelines.show()

Below is the first instance of a line chart being broken up due to being too big. The logic I had with these broken up figures was that the term total will be the "default" state, and then breaking up the figures into segments that are digestible by PowerBI and still flow well. Most were able to fit into a logical range (i.e. 10000-20000), but others get weird (namely THEA - it gets broken up more just due to how many courses, and thus traces/figures, there are)

In [458]:
#Section 3k(a) - MDJ Term Total Line Chart

#Since this is just looking at the term total, do not need the sorting logic from before
coursepct = mdjmtline[mdjmtline["Course Code"] == "MDJ - Term Total"] 

mdjtotalrateline = go.Figure()

mdjtotalrateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

mdjtotalrateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

mdjtotalrateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

mdjtotalrateline.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

mdjtotalrateline.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>MDJ Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

mdjtotalrateline.show()

In [459]:
#Section 3k(b) - MDJ 20k and lower Line Chart

#sets the criteria for the specific courses we are looking at
mdjmtlineunder20k = mdjmtline[mdjmtline["Course"] < 20000]

#Similar to above methods of sorting, but cutting out the term totals. We keep the MDJ- Term Total so it doesn't get picked up
other_courses = (
    mdjmtlineunder20k[mdjmtlineunder20k["Course Code"] != "MDJ - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)

courses = other_courses
coursebuttons = []
mdjunder20krateline = go.Figure()

for i, course in enumerate(courses): 
    coursepct = mdjmtlineunder20k[mdjmtlineunder20k["Course Code"] == course] 
    is_visible = (i == 0)

    mdjunder20krateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    mdjunder20krateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    mdjunder20krateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    mdjunder20krateline.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    mdjunder20krateline.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>MDJ Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .4,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

mdjunder20krateline.show()

In [460]:
#Section 3k(c) - MDJ 20k - 25k Line Chart

mdjmtline20kto25k = mdjmtline[(mdjmtline["Course"] > 20000) & (mdjmtline["Course"] < 25000)]


other_courses = (
    mdjmtline20kto25k[mdjmtline20kto25k["Course Code"] != "MDJ - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = other_courses

coursebuttons = []

mdj20kto25kratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = mdjmtline20kto25k [mdjmtline20kto25k ["Course Code"] == course] 
    is_visible = (i == 0)

    mdj20kto25kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    mdj20kto25kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    mdj20kto25kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    mdj20kto25kratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    mdj20kto25kratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>MDJ Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .405,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

mdj20kto25kratelines.show()

In [461]:
#Section 3k(a) - MDJ 25k+ Line Chart

mdjmtlineabove25k = mdjmtline[mdjmtline["Course"] > 25000]

other_courses = (
    mdjmtlineabove25k[mdjmtlineabove25k["Course Code"] != "MDJ - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = other_courses

coursebuttons = []

mdjabove25krateline = go.Figure()

for i, course in enumerate(courses): 
    coursepct = mdjmtlineabove25k[mdjmtlineabove25k["Course Code"] == course] 
    is_visible = (i == 0)

    mdjabove25krateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    mdjabove25krateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    mdjabove25krateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    mdjabove25krateline.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    mdjabove25krateline.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>MDJ Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .405,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

mdjabove25krateline.show()

In [462]:
#Section 3l(a) - VCD Total Term Line Chart

coursepct = vcdmtline[vcdmtline["Course Code"] == "VCD - Term Total"] 

vcdtotalrateline = go.Figure()

vcdtotalrateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

vcdtotalrateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

vcdtotalrateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

vcdtotalrateline.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

vcdtotalrateline.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>VCD Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

vcdtotalrateline.show()

In [463]:
#Section 3l(b) - VCD 10-20k Line Chart

vcdmtlinebelow20k = vcdmtline[vcdmtline["Course"] < 20000]

other_courses = (
    vcdmtlinebelow20k[vcdmtlinebelow20k["Course Code"] != "VCD - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = other_courses
coursebuttons = []

vcdunder20kratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = vcdmtlinebelow20k[vcdmtlinebelow20k["Course Code"] == course] 
    is_visible = (i == 0)

    vcdunder20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    vcdunder20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    vcdunder20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    vcdunder20kratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    vcdunder20kratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>VCD Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .405,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

vcdunder20kratelines.show()

In [464]:
#Section 3l(c) - VCD 20k+ Line Chart

vcdmtlineabove20k = vcdmtline[vcdmtline["Course"] > 20000]

other_courses = (
    vcdmtlineabove20k[vcdmtlineabove20k["Course Code"] != "VCD - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = other_courses
coursebuttons = []

vcdabove20kratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = vcdmtlineabove20k[vcdmtlineabove20k["Course Code"] == course] 
    is_visible = (i == 0)

    vcdabove20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    vcdabove20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    vcdabove20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    vcdabove20kratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    vcdabove20kratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>VCD Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .405,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

vcdabove20kratelines.show()

In [465]:
#Section 3m(a) - ART ART Line Chart

artsubmtline = artmtline[artmtline["Subject"] == "ART"]

term_total = ["ART - Term Total"] 
other_courses = (
    artsubmtline[artsubmtline["Course Code"] != "ART - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)

courses = term_total + other_courses
coursebuttons = []

artsubratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = artsubmtline[artsubmtline["Course Code"] == course] 
    is_visible = (i == 0)

    artsubratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    artsubratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    artsubratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    artsubratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    artsubratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>ART Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .385,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

artsubratelines.show()

In [466]:
#Section 3m(b) - ART ARTH Line Chart

arthsubmtline = artmtline[artmtline["Subject"] == "ARTH"]

term_total = ["ARTH - Term Total"] 
other_courses = (
    arthsubmtline[arthsubmtline["Course Code"] != "ARTH - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = term_total + other_courses
coursebuttons = []

arthsubratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = arthsubmtline[arthsubmtline["Course Code"] == course] 
    is_visible = (i == 0)

    arthsubratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    arthsubratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    arthsubratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    arthsubratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    arthsubratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>ARTH Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .385,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

arthsubratelines.show()

In [467]:
#Section 3m(c) - ART ARTS Line Chart

artssubmtline = artmtline[artmtline["Subject"] == "ARTS"]

term_total = ["ARTS - Term Total"] 
other_courses = (
    artssubmtline[artssubmtline["Course Code"] != "ARTS - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = term_total + other_courses
coursebuttons = []

artssubratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = artssubmtline[artssubmtline["Course Code"] == course] 
    is_visible = (i == 0)

    artssubratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    artssubratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    artssubratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    artssubratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    artssubratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>ARTS Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .385,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

artssubratelines.show()

In [468]:
#Section 3n (a) - MUS Term Total Line Chart

coursepct = musmtline[musmtline["Course Code"] == "MUS - Term Total"] 

mustotalrateline = go.Figure()

mustotalrateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

mustotalrateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

mustotalrateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

mustotalrateline.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

mustotalrateline.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>MUS Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

mustotalrateline.show()

In [469]:
#Section 3n(b) - MUS 10-20k Line Chart

musmtlinebelow20k = musmtline[musmtline["Course"] < 20000]

other_courses = (
    musmtlinebelow20k[musmtlinebelow20k["Course Code"] != "MUS - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = other_courses
coursebuttons = []

musbelow20kratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = musmtlinebelow20k[musmtlinebelow20k["Course Code"] == course] 
    is_visible = (i == 0)

    musbelow20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    musbelow20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    musbelow20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    musbelow20kratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    musbelow20kratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>MUS Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .405,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

musbelow20kratelines.show()

In [470]:
#Section 3n(c) - MUS 20k+ Line Chart

musmtlineabove20k = musmtline[musmtline["Course"] > 20000]

other_courses = (
    musmtlineabove20k[musmtlineabove20k ["Course Code"] != "MUS - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses =  other_courses

coursebuttons = []

musabove20kratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = musmtlineabove20k[musmtlineabove20k["Course Code"] == course] 
    is_visible = (i == 0)

    musabove20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    musabove20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    musabove20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    musabove20kratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    musabove20kratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>MUS Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .405,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

musabove20kratelines.show()

In [471]:
#Section 3o (a) - FDM Term Total Line Chart

coursepct = fdmmtline[fdmmtline["Course Code"] == "FDM - Term Total"] 

fdmtotalrateline = go.Figure()

fdmtotalrateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

fdmtotalrateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

fdmtotalrateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

fdmtotalrateline.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

fdmtotalrateline.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>FDM Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

fdmtotalrateline.show()

In [472]:
#Section 3o(b) - FDM 10-20k Line Chart
fdmmtlinebelow20k = fdmmtline[fdmmtline["Course"] < 20000]

other_courses = (
   fdmmtlinebelow20k[fdmmtlinebelow20k["Course Code"] != "FDM - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = other_courses

coursebuttons = []

fdmbelow20kratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = fdmmtlinebelow20k[fdmmtlinebelow20k["Course Code"] == course] 
    is_visible = (i == 0)

    fdmbelow20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    fdmbelow20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    fdmbelow20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    fdmbelow20kratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    fdmbelow20kratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>FDM Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .405,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

fdmbelow20kratelines.show()

In [473]:
#Section 3o(c) - FDM 20k+ Line Chart
fdmmtlineabove20k = fdmmtline[fdmmtline["Course"] > 20000]

other_courses = (
    fdmmtlineabove20k[fdmmtlineabove20k["Course Code"] != "FDM - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = other_courses

coursebuttons = []

fdmabove20kratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = fdmmtlineabove20k[fdmmtlineabove20k["Course Code"] == course] 
    is_visible = (i == 0)

    fdmabove20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    fdmabove20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    fdmabove20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    fdmabove20kratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    fdmabove20kratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>FDM Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .405,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

fdmabove20kratelines.show()

In [474]:
#Section 3p (a) - THDN DAN Subject Line Chart

dansubmtline = thdnmtline[thdnmtline["Subject"] == "DAN"]


term_total = ["DAN - Term Total"] 
other_courses = (
    dansubmtline[dansubmtline["Course Code"] != "DAN - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = term_total + other_courses

coursebuttons = []

dansubratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = dansubmtline[dansubmtline["Course Code"] == course] 
    is_visible = (i == 0)

    dansubratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    dansubratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    dansubratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    dansubratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    dansubratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>DAN Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .385,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

dansubratelines.show()

In [475]:
#Section 3p (b) - THEA Term Total Line Chart

theasubmtline = thdnmtline[thdnmtline["Subject"] == "THEA"]

coursepct = theasubmtline[theasubmtline["Course Code"] == "THEA - Term Total"] 

theatotalrateline = go.Figure()

theatotalrateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

theatotalrateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

theatotalrateline.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

theatotalrateline.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

theatotalrateline.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>THEA Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

theatotalrateline.show()

In [476]:
#Section 3p - THDN Below 14k Line Chart
theasubmtline = thdnmtline[thdnmtline["Subject"] == "THEA"]
theamtlinebelow14k = theasubmtline[theasubmtline["Course"] <14000]

other_courses = (
    theamtlinebelow14k[theamtlinebelow14k["Course Code"] != "THEA - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = other_courses

coursebuttons = []

theabelow14kratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = theamtlinebelow14k[theamtlinebelow14k["Course Code"] == course] 
    is_visible = (i == 0)

    theabelow14kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    theabelow14kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    theabelow14kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    theabelow14kratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    theabelow14kratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>THEA Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .405,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

theabelow14kratelines.show()

In [477]:
#Section 3p - THDN 14k - 20k Line Chart
theasubmtline = thdnmtline[thdnmtline["Subject"] == "THEA"]
theamtline14kto20k = theasubmtline[(theasubmtline["Course"] > 14000) & (theasubmtline["Course"] < 20000)]

other_courses = (
    theamtline14kto20k[theamtline14kto20k["Course Code"] != "THEA - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = other_courses

coursebuttons = []

thea14kto20kratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = theamtline14kto20k[theamtline14kto20k["Course Code"] == course] 
    is_visible = (i == 0)

    thea14kto20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    thea14kto20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    thea14kto20kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    thea14kto20kratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    thea14kto20kratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>THEA Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .405,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

thea14kto20kratelines.show()

In [478]:
#Section 3p - THDN 20-25k Line Chart
theasubmtline = thdnmtline[thdnmtline["Subject"] == "THEA"]
theamtline20kto25k = theasubmtline[(theasubmtline["Course"] > 20000) & (theasubmtline["Course"] < 25000)]

other_courses = (
    theamtline20kto25k[theamtline20kto25k["Course Code"] != "THEA - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = other_courses

coursebuttons = []

thea20kto25kratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = theamtline20kto25k[theamtline20kto25k["Course Code"] == course] 
    is_visible = (i == 0)

    thea20kto25kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    thea20kto25kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    thea20kto25kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    thea20kto25kratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    thea20kto25kratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>THEA Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .405,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

thea20kto25kratelines.show()

In [479]:
#Section 3p - THDN 25-30k Line Chart
theasubmtline = thdnmtline[thdnmtline["Subject"] == "THEA"]
theamtlineabove25k = theasubmtline[theasubmtline["Course"] > 25000]

other_courses = (
    theamtlineabove25k[theamtlineabove25k["Course Code"] != "THEA - Term Total"]
    .drop_duplicates(subset=["Course Code"])
    .sort_values("Course")["Course Code"]
    .tolist()
)
courses = other_courses

coursebuttons = []

theaabove25kratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = theamtlineabove25k[theamtlineabove25k["Course Code"] == course] 
    is_visible = (i == 0)

    theaabove25kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Passing Grades"].tolist(),
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"].tolist(), 
        hovertemplate = (
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    theaabove25kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Failing Grades"].tolist(),
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"].tolist(),
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    theaabove25kratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"].tolist(),
        y = coursepct["# of Unreported Grades"].tolist(),
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"].tolist(),
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    theaabove25kratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"].astype(int).tolist(),
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"].tolist(),
        y = [-10] * len(coursepct["Academic Period"]),
        customdata = coursepct["Academic Period"],
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
        hovertemplate = (
            "<b>Count of Enrolled Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{text} students"\
            "<extra></extra>"
        )
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    theaabove25kratelines.update_layout(
        font_family = "Segoe UI",
            xaxis = dict(
            title = dict(text = "<i>Semester</i>", font_size = 18),
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>", font_size = 18),
        ),
        title=dict(text= "<b>THEA Midterm Grade Outcomes Over Time</b>",
            font_size = 24, 
            xanchor = "center",
            x = .5,
            yanchor = "top",
            y = .95,
            font_color = "black"),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .51,
            y = 1.17,
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "<b>Legend:</b>", font_color = "black", font_size = 16),
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, 
            orientation = "h" 
        ),
        annotations=[dict(
        text = "<b>Course:</b>", 
        font_size = 16,
        showarrow = False,
        xref = "paper",
        x = .405,
        yref = "paper",
        y = 1.155,
        font_color = "black"
    )],
        shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-15, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below",
        )
    ],
        )

theaabove25kratelines.show()

#### <center>**Section 4 - Intervention Table**</center>

The code in this section is focused on creating multiple tables that summarize how many students require an intervention each semester. Through this table, college leadership can see how many students will be contacted by an advisor. 

Note that this table looks at only students who qualify for an intervention - hub students who passed and do not qualify for an intervention and all non-hub students are not included in this table.

##### <center>*Reloading the Data*</center>

In [480]:
#Section 4a - Reloading the Data
hubinterventions = pd.read_csv("mtinterventions.csv")
hubinterventions.head()

,Unnamed: 0,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Mid Term Grade Number,Final Grade Number,MT Intervention Needed
0,33,33,Registered,AED,22860,KC,B-,C-,ARCH,ID,202280,FR,AED 22860,CAED,CAED,1.7,2.7,FR/SO Intervention Needed
1,31,31,Std Withdrawn,AED,22860,KC,W,D,ARCH,ID,202280,SO,AED 22860,CAED,CAED,1.0,0.0,FR/SO Intervention Needed
2,11,11,Registered,AED,22860,KC,NaN,C,ARCH,ID,202280,FR,AED 22860,CAED,CAED,2.0,NaN,FR/SO Intervention Needed
3,78,78,Std Withdrawn,AED,22860,KC,W,F,ARCH,ID,202280,FR,AED 22860,CAED,CAED,0.0,0.0,FR/SO Intervention Needed
4,69,69,Registered,AED,22860,KC,B-,C,ARCH,ID,202280,FR,AED 22860,CAED,CAED,2.0,2.7,FR/SO Intervention Needed


##### <center>*Creating Intervention Summary Tables*</center>

Using the same logic as before, we are just gathering up each instance and making it digestable - the tables are just concerned with the total amount of interventions, so we do not need to get as picky with breaking down the subjects.

In [481]:
#Section 4b - Making the Summary Table
hubinterventions["MT Intervention Needed"].value_counts()
hubpivot = hubinterventions.pivot_table(index = ["Academic Period", "Major College"],
                            columns = "MT Intervention Needed",
                            values = "Record ID",
                            aggfunc = "count").reset_index().rename_axis(None, axis=1)

hubpivot.head()

,Academic Period,Major College,FR/SO Intervention Needed,JR/SR Intervention Needed
0,202280,CAED,133,15
1,202280,CCI,209,49
2,202280,CotA,424,37
3,202310,CAED,122,18
4,202310,CCI,155,32


##### <center>*Adding a Total Instances Column*</center>

In [482]:
#Section 4c - Creating a Total Column
hubpivot["Total # of Instances"] = hubpivot["FR/SO Intervention Needed"] + hubpivot["JR/SR Intervention Needed"]
hubpivot.head()

,Academic Period,Major College,FR/SO Intervention Needed,JR/SR Intervention Needed,Total # of Instances
0,202280,CAED,133,15,148
1,202280,CCI,209,49,258
2,202280,CotA,424,37,461
3,202310,CAED,122,18,140
4,202310,CCI,155,32,187


##### <center>*Making College-Level Dataframes*</center>

As advisors are looking at the college level, we do not need to create individual major/subject tables for looking at students.

In [483]:
#Section 4d - Creating College-Level Dataframes
caedpivot = hubpivot[(hubpivot["Major College"] == "CAED") & (hubpivot["Academic Period"].isin(currentterm))]
ccipivot = hubpivot[(hubpivot["Major College"] == "CCI")  & (hubpivot["Academic Period"].isin(currentterm))]
cotapivot = hubpivot[(hubpivot["Major College"] == "CotA")  & (hubpivot["Academic Period"].isin(currentterm))]

##### <center>*Creating Advisor Intervention Tables*</center>

In [484]:
#Section 4e - CAED Intervention Table
caedintertrace = go.Table( #we are primarily following the style guides set out earlier - but with a few new twists
    header=dict(values = ["Group #", "CAED Midterm Intervention - Student Criteria", "# of Instances"],                
                line_color = "black",
                fill_color = "#EFAB00",
                font_color = "black",
                font_weight = "bold"),
    cells = dict(values = [
                [1,2," "],
                 ["CAED Major, FR or SO, CAED Course, midterm D+ or below, still registered", 
                  "CAED Major, JR or SR, CAED Course, midterm F, still registered",
                  "Total"],
                 [caedpivot["FR/SO Intervention Needed"], caedpivot["JR/SR Intervention Needed"], caedpivot["Total # of Instances"]]],
                 line_color = "black",
                 #For the below, we have to specifically call out the colors. This helps us define the row colors, but for the last row it helps it pop much more.
                 fill_color = [["#FFFFFF", "#FFF1CC", "#EFAB00"], ["#FFFFFF", "#FFF1CC", "#EFAB00"], ["#FFFFFF", "#FFF1CC", "#EFAB00"]],
                 font_color = "black"),
    visible = True)

interlayout = go.Layout(
    title=dict(text=f"<b>{prettyterm} CAED Advisor Intervention Table",
            font_weight = "bold", 
            xanchor = "center",
            x = .5,
            font_color = "black",
            font_size = 24),
    font_family = "Segoe UI",
    width = 700,
    height = 280,
    margin = dict(l = 20, r = 20, b = 20, t = 80) #This helps tighten the figure up for posting on PowerBI
)

caedintertab = go.Figure(data = caedintertrace, layout=interlayout)

caedintertab.show()

In [485]:
#Section 4f - CCI Intervention Table
cciintertrace = go.Table(
    header=dict(values = ["Group #", "CCI Midterm Intervention - Student Criteria", "# of Instances"],                
                line_color = "black",
                fill_color = "#EFAB00",
                font_color = "black",
                font_weight = "bold"),
    cells = dict(values = [
                [1,2," "],
                 ["CCI Major, FR or SO, CAED Course, midterm D+ or Below, still registered", 
                  "CCI Major, JR or SR, CAED Course, midterm F, still registered",
                  "Total"],
                 [ccipivot["FR/SO Intervention Needed"], ccipivot["JR/SR Intervention Needed"], ccipivot["Total # of Instances"]]],
                 line_color = "black",
                 fill_color = [["#FFFFFF", "#FFF1CC", "#EFAB00"], ["#FFFFFF", "#FFF1CC", "#EFAB00"], ["#FFFFFF", "#FFF1CC", "#EFAB00"]],
                 font_color = "black"),
    visible = True)

interlayout = go.Layout(
    title=dict(text=f"<b>{prettyterm} CCI Advisor Intervention Table",
            font_weight = "bold", 
            xanchor = "center",
            x = .5,
            font_color = "black",
            font_size = 24),
    width = 700,
    height = 280,
    margin = dict(l = 20, r = 20, b = 20, t = 80),
    font_family = "Segoe UI"
)

cciintertab = go.Figure(data = cciintertrace, layout=interlayout)

cciintertab.show()

In [486]:
#Section 4g - CotA Intervention Table
cotaintertrace = go.Table(
    header=dict(values = ["Group #", "CotA Midterm Intervention - Student Criteria", "# of Instances"],                
                line_color = "black",
                fill_color = "#EFAB00",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values = [
                [1,2," "],
                 ["CotA Major, FR or SO, CotA Course, midterm D+ or Below, still registered", 
                  "CotA Major, JR or SR, CotA Course, midterm F, still registered",
                  "Total"],
                 [cotapivot["FR/SO Intervention Needed"], cotapivot["JR/SR Intervention Needed"], cotapivot["Total # of Instances"]]],
                 line_color = "black",
                 fill_color = [["#FFFFFF", "#FFF1CC", "#EFAB00"], ["#FFFFFF", "#FFF1CC", "#EFAB00"], ["#FFFFFF", "#FFF1CC", "#EFAB00"]],
                 font_color = "black"),
    visible = True)

interlayout = go.Layout(
    title=dict(text=f"<b>{prettyterm} CotA Advisor Intervention Table",
            font_weight = "bold", 
            xanchor = "center",
            x = .5,
            font_color = "black",
            font_size = 24),
    width = 700,
    height = 280,
    margin = dict(l = 20, r = 20, b = 20, t = 80),
    font_family = "Segoe UI"
)

cotaintertab = go.Figure(data = cotaintertrace, layout=interlayout)

cotaintertab.show()

#### <center>**Section 5 - Intervention Over Time**</center>

The code in this section is focused on the creation of a line chart that shows the total number of interventions needed over time. Similar to the tables above, it is focused at the college-level, not school level. A majority of the data "cleaning" was finished in the last section - the only cleaning needed is to sort the terms and replace them with the written out version of the code.

##### <center>*Sorting & Mapping Terms*</center>

Since we have done some moving around of terms due to the pivoting, we are just resorting the frame so it looks nice. We are also mapping the academic periods so it is clear which term(s) we are looking at, as opposed to referencing the term codes (which not everyone knows).

In [487]:
#Section 5a - Sorting Terms
hubinterventionlines = hubpivot.sort_values("Academic Period")
hubinterventionlines

,Academic Period,Major College,FR/SO Intervention Needed,JR/SR Intervention Needed,Total # of Instances
0,202280,CAED,133,15,148
1,202280,CCI,209,49,258
2,202280,CotA,424,37,461
3,202310,CAED,122,18,140
4,202310,CCI,155,32,187
5,202310,CotA,392,35,427
6,202380,CAED,157,6,163
7,202380,CCI,210,26,236
8,202380,CotA,442,30,472
9,202410,CAED,115,28,143


In [488]:
#Section 5b - Mapping Term Names to Numbers
hubinterventionlines["Academic Period"] = hubinterventionlines["Academic Period"].map(figtermmap)
hubinterventionlines["Academic Period"].unique()

<StringArray>
[  'Fall 2022', 'Spring 2023',   'Fall 2023', 'Spring 2024',   'Fall 2024',
 'Spring 2025',   'Fall 2025', 'Spring 2026']
Length: 8, dtype: str

##### <center>*Creating College-Specific Dataframes*</center>

In [489]:
#Section 5c - Creating College-Specific Frames
caedinterline = hubinterventionlines[hubinterventionlines["Major College"] == "CAED"]
cciinterline = hubinterventionlines[hubinterventionlines["Major College"] == "CCI"]
cotainterline = hubinterventionlines[hubinterventionlines["Major College"] == "CotA"]

##### <center>*Creating the Intervention Figures*</center>

In [490]:
#Section 5d - CAED Figure
frsotrace = go.Scatter(
    x = caedinterline["Academic Period"].tolist(),
    y = caedinterline["FR/SO Intervention Needed"].tolist(),
    name = "FR/SO Interventions Needed",
    line = dict(color = "#003976", width = 2), #Since there isn't a biological color for FR/SO or JR/SR, I'm just using the institutional colors for the lines
    #For below - this helps highlight the change in FR/SO criteria for interventions that started in Spring 2026, while past terms have the old criteria
    customdata = ["Criteria: MT Grade of D+ or lower" if x == "Spring 2026" else "Criteria: MT Grade of C or below" for x in caedinterline["Academic Period"]],
    hovertemplate = (
    "<b>FR/SO Interventions</b>"\
    "<br>Term: %{x}"\
    "<br># of Interventions Needed: %{y}"\
    "<br>%{customdata}"\
    "<extra></extra>")) 

jrsrtrace = go.Scatter(
    x = caedinterline["Academic Period"].tolist(),
    y = caedinterline["JR/SR Intervention Needed"].tolist(),
    name = "JR/SR Interventions Needed",
    line = dict(color = "#EFAB00", width = 2),
    hovertemplate = (
    "<b>JR/SR Interventions</b>"\
    "<br>Term: %{x}"\
    "<br># of Interventions Needed: %{y}"\
    "<br>Criteria: MT Grade of F"
    "<extra></extra>"))


internum = go.Scatter( #Using the same logic as before - showing the total number of interventions. This helps give a quick "at a glance" of how many interventions are needed each term
    mode = "text",
    text = caedinterline["Total # of Instances"].astype(int).tolist(),
    textfont = dict(weight = "bold"),
    x = caedinterline["Academic Period"].tolist(),
    y = [-12] * len(caedinterline["Academic Period"].tolist()),
    name = "Total Interventions (n)",
    showlegend = (True),
            hovertemplate = (
            "<b>Total Number of Interventions</b>"\
            "<br>Term: %{x}"\
            "<br># of Interventions: %{text}"\
            "<extra></extra>"
        )
)

layout = go.Layout(
    title = dict(text = "<b>CAED Interventions Needed Over Time</b>",
        font = dict(size = 24, color = "black"),
        xanchor = "center",
        x = 0.5,
        yanchor = "top", 
        y = 0.93
    ),
    xaxis = dict(
        title = dict(text = "<i>Semester</i>", font = dict(size = 18)
                     )
    ),
    yaxis = dict(
        title = dict(text = "<i># of Intervention</i>", 
        font = dict(size = 18)),
        range = [-20, 200], #This range will vary by subject. It needs set (especially the lower end) to allow the text to sit on the figure. The higher end needs changed to accomodate the variety of interventions needed.
        dtick = 50
    ),
    legend = dict(
        title = dict(
        text = "<b>Legend:</b>",
        font = dict(color = "black")),
        xanchor = "center",
        x = 0.5,
        yanchor = "bottom", 
        y = -0.4,
        orientation = "h" 
    ),
    font = dict(family = "Segoe UI")
)

caedinterlines = go.Figure(data = [frsotrace, jrsrtrace, internum], layout = layout)

caedinterlines.update_layout(
            shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-19, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below"
        )
    ],
    margin= dict(l = 20, r = 20, b = 20, t = 80), #This helps tighten the figure up for posting on PowerBI
    height = 350,
    width = 1200
)
caedinterlines.add_annotation(
    x= 7,
    y = 84,
    ax = 0,
    ay = -40,
    font = dict(size = 10),
    text = "Change in"\
         "<br>FR/SO Criteria",
)

caedinterlines.show()

In [491]:
#Section 5e - CCI Figure
frsotrace = go.Scatter(
    x = cciinterline["Academic Period"].tolist(),
    y = cciinterline["FR/SO Intervention Needed"].tolist(),
    name = "FR/SO Interventions Needed",
    line = dict(color = "#003976", width = 2),
    customdata = ["Criteria: MT Grade of D+ or lower" if x == "Spring 2026" else "Criteria: MT Grade of C or below" for x in cciinterline["Academic Period"]],
    hovertemplate = (
    "<b>FR/SO Interventions</b>"\
    "<br>Term: %{x}"\
    "<br># of Interventions Needed: %{y}"\
    "<br>%{customdata}"\
    "<extra></extra>")) 

jrsrtrace = go.Scatter(
    x = cciinterline["Academic Period"].tolist(),
    y = cciinterline["JR/SR Intervention Needed"].tolist(),
    name = "JR/SR Interventions Needed",
    line = dict(color = "#EFAB00", width = 2),
    hovertemplate = (
    "<b>JR/SR Interventions</b>"\
    "<br>Term: %{x}"\
    "<br># of Interventions Needed: %{y}"\
    "<br>Criteria: MT Grade of F"
    "<extra></extra>"))


internum = go.Scatter(
    mode = "text",
    text = cciinterline["Total # of Instances"].astype(int).tolist(),
    textfont = dict(weight = "bold"),
    x = cciinterline["Academic Period"].tolist(),
    y = [-12] * len(cciinterline["Academic Period"].tolist()),
    name = "Total Interventions (n)",
    showlegend = (True),
            hovertemplate = (
            "<b>Total Number of Interventions</b>"\
            "<br>Term: %{x}"\
            "<br># of Interventions: %{text}"\
            "<extra></extra>"
        )
)

layout = go.Layout(
    title = dict(text = "<b>CCI Interventions Needed Over Time</b>",
        font = dict(size = 24, color = "black"),
        xanchor = "center",
        x = 0.5,
        yanchor = "top", 
        y = 0.93
    ),
    xaxis = dict(
        title = dict(text = "<i>Semester</i>", font = dict(size = 18)
                     )
    ),
    yaxis = dict(
        title = dict(text = "<i># of Intervention</i>", 
        font = dict(size = 18)),
        range = [-20, 250],
        dtick = 50
    ),
    legend = dict(
        title = dict(
        text = "<b>Legend:</b>",
        font = dict(color = "black")),
        xanchor = "center",
        x = 0.5,
        yanchor = "bottom", 
        y = -0.4,
        orientation = "h" 
    ),
    font = dict(family = "Segoe UI")
)

cciinterlines = go.Figure(data = [frsotrace, jrsrtrace, internum], layout = layout)

cciinterlines.update_layout(
            shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-19, 
            y1=-3,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below"
        )
    ],
    margin= dict(l = 20, r = 20, b = 20, t = 80),
    height = 350,
    width = 1200
)
cciinterlines.add_annotation(
    x= 7,
    y = 72,
    ax = 0,
    ay = -40,
    font = dict(size = 10),
    text = "Change in"\
         "<br>FR/SO Criteria",
)

cciinterlines.show()

In [492]:
#Section 5e - CotA Figure
frsotrace = go.Scatter(
    x = cotainterline["Academic Period"].tolist(),
    y = cotainterline["FR/SO Intervention Needed"].tolist(),
    name = "FR/SO Interventions Needed",
    line = dict(color = "#003976", width = 2),
    customdata = ["Criteria: MT Grade of D+ or lower" if x == "Spring 2026" else "Criteria: MT Grade of C or below" for x in cotainterline["Academic Period"]],
    hovertemplate = (
    "<b>FR/SO Interventions</b>"\
    "<br>Term: %{x}"\
    "<br># of Interventions Needed: %{y}"\
    "<br>%{customdata}"\
    "<extra></extra>")) 

jrsrtrace = go.Scatter(
    x = cotainterline["Academic Period"].tolist(),
    y = cotainterline["JR/SR Intervention Needed"].tolist(),
    name = "JR/SR Interventions Needed",
    line = dict(color = "#EFAB00", width = 2),
    hovertemplate = (
    "<b>JR/SR Interventions</b>"\
    "<br>Term: %{x}"\
    "<br># of Interventions Needed: %{y}"\
    "<br>Criteria: MT Grade of F"
    "<extra></extra>"))


internum = go.Scatter(
    mode = "text",
    text = cotainterline["Total # of Instances"].astype(int).tolist(),
    textfont = dict(weight = "bold"),
    x = cotainterline["Academic Period"].tolist(),
    y = [-5] * len(cotainterline["Academic Period"].tolist()),
    name = "Total Interventions (n)",
    showlegend = (True),
            hovertemplate = (
            "<b>Total Number of Interventions</b>"\
            "<br>Term: %{x}"\
            "<br># of Interventions: %{text}"\
            "<extra></extra>"
        )
)

layout = go.Layout(
    title = dict(text = "<b>CotA Interventions Needed Over Time</b>",
        font = dict(size = 24, color = "black"),
        xanchor = "center",
        x = 0.5,
        yanchor = "top", 
        y = 0.93
    ),
    xaxis = dict(
        title = dict(text = "<i>Semester</i>", font = dict(size = 18)
                     )
    ),
    yaxis = dict(
        title = dict(text = "<i># of Intervention</i>", 
        font = dict(size = 18)),
        range = [-20, 475],
        dtick = 75
    ),
    legend = dict(
        title = dict(
        text = "<b>Legend:</b>",
        font = dict(color = "black")),
        xanchor = "center",
        x = 0.5,
        yanchor = "bottom", 
        y = -0.4,
        orientation = "h" 
    ),
    font = dict(family = "Segoe UI")
)

cotainterlines = go.Figure(data = [frsotrace, jrsrtrace, internum], layout = layout)

cotainterlines.update_layout(
            shapes=[dict(
            type="rect",
            xref="paper", 
            yref="y",
            x0=0,
            x1=1,
            y0=-19, 
            y1=15,
            fillcolor="white",
            line=dict(width=1, color = "black"),
            layer="below"
        )
    ],
    margin= dict(l = 20, r = 20, b = 20, t = 80),
    height = 350,
    width = 1200
)
cotainterlines.add_annotation(
    x= 7,
    y = 268,
    ax = 0,
    ay = -40,
    font = dict(size = 10),
    text = "Change in"\
         "<br>FR/SO Criteria",
)

cotainterlines.show()

#### <center>**Section 6 - Intervention Contact Sheet**</center>

The code in this section is dedicated to the creation of the intervention contact sheet for academic advisors. This sheet will allow advisors to know who to contact, their contact information and what course(s) indicate a failing MT grade. as the advisors operate under a hub model, a single Excel file will be created with tabs for each college. The advising team may download this sheet and create a shared copy that they can reference to track which student(s) have been contacted.

Students will be assigned into groups based on their status. The hierarchy has been previously been determined by the advising team. In the current iteration of the file, there are only two groups for each college. In future iterations, students will be broken into additional groups based on major, special course requirements and more.

##### <center>*Reloading the Data*</center>

In [493]:
#Section 6a - Reloading the Data
hubcontacts = pd.read_csv("mtinterventions.csv")
hubcontacts.head()

,Unnamed: 0,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Mid Term Grade Number,Final Grade Number,MT Intervention Needed
0,33,33,Registered,AED,22860,KC,B-,C-,ARCH,ID,202280,FR,AED 22860,CAED,CAED,1.7,2.7,FR/SO Intervention Needed
1,31,31,Std Withdrawn,AED,22860,KC,W,D,ARCH,ID,202280,SO,AED 22860,CAED,CAED,1.0,0.0,FR/SO Intervention Needed
2,11,11,Registered,AED,22860,KC,NaN,C,ARCH,ID,202280,FR,AED 22860,CAED,CAED,2.0,NaN,FR/SO Intervention Needed
3,78,78,Std Withdrawn,AED,22860,KC,W,F,ARCH,ID,202280,FR,AED 22860,CAED,CAED,0.0,0.0,FR/SO Intervention Needed
4,69,69,Registered,AED,22860,KC,B-,C,ARCH,ID,202280,FR,AED 22860,CAED,CAED,2.0,2.7,FR/SO Intervention Needed


##### <center>*Assigning Group Numbers to Students*</center>

In [494]:
#Section 6b - Assigning Students to Groups
#Note that the "group" criteria is preset (1, 2) - this is what will be expanded on once we have the "real" data
hubcontacts["MT Intervention Group"] = np.where(hubcontacts["MT Intervention Needed"] == "FR/SO Intervention Needed", 1, 2)
hubcontacts.head()

,Unnamed: 0,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Mid Term Grade Number,Final Grade Number,MT Intervention Needed,MT Intervention Group
0,33,33,Registered,AED,22860,KC,B-,C-,ARCH,ID,202280,FR,AED 22860,CAED,CAED,1.7,2.7,FR/SO Intervention Needed,1
1,31,31,Std Withdrawn,AED,22860,KC,W,D,ARCH,ID,202280,SO,AED 22860,CAED,CAED,1.0,0.0,FR/SO Intervention Needed,1
2,11,11,Registered,AED,22860,KC,NaN,C,ARCH,ID,202280,FR,AED 22860,CAED,CAED,2.0,NaN,FR/SO Intervention Needed,1
3,78,78,Std Withdrawn,AED,22860,KC,W,F,ARCH,ID,202280,FR,AED 22860,CAED,CAED,0.0,0.0,FR/SO Intervention Needed,1
4,69,69,Registered,AED,22860,KC,B-,C,ARCH,ID,202280,FR,AED 22860,CAED,CAED,2.0,2.7,FR/SO Intervention Needed,1


In [495]:
#Section 6c - Adding Advisor Entry Notes
#These sections are to allow advisors to manually enter notes from the meeting with students
hubcontacts[["Notes", "Advising Pin", "Hold", "Advisor", "Date"]] = np.nan
hubcontacts.head()

,Unnamed: 0,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,...,Subject College,Mid Term Grade Number,Final Grade Number,MT Intervention Needed,MT Intervention Group,Notes,Advising Pin,Hold,Advisor,Date
0,33,33,Registered,AED,22860,KC,B-,C-,ARCH,ID,...,CAED,1.7,2.7,FR/SO Intervention Needed,1,NaN,NaN,NaN,NaN,NaN
1,31,31,Std Withdrawn,AED,22860,KC,W,D,ARCH,ID,...,CAED,1.0,0.0,FR/SO Intervention Needed,1,NaN,NaN,NaN,NaN,NaN
2,11,11,Registered,AED,22860,KC,NaN,C,ARCH,ID,...,CAED,2.0,NaN,FR/SO Intervention Needed,1,NaN,NaN,NaN,NaN,NaN
3,78,78,Std Withdrawn,AED,22860,KC,W,F,ARCH,ID,...,CAED,0.0,0.0,FR/SO Intervention Needed,1,NaN,NaN,NaN,NaN,NaN
4,69,69,Registered,AED,22860,KC,B-,C,ARCH,ID,...,CAED,2.0,2.7,FR/SO Intervention Needed,1,NaN,NaN,NaN,NaN,NaN


##### <center>*Formatting Table to Final Version*</center>


In [496]:
#Section 6d - Formatting Table to the Final Version
hubcontacts = hubcontacts.drop(columns = ["Unnamed: 0", 'Final Grade', "Subject College", 'Final Grade Number', 'Mid Term Grade Number', 'MT Intervention Needed', 'Registration Status','Campus'], axis=0) #Dropping Unnecessary Columns
hubcontacts = hubcontacts.rename(columns = {"Record ID":"ID"}) #Replacing Record ID w/ ID
hubcontacts = hubcontacts[['MT Intervention Group','Notes','Advising Pin','Hold','Advisor', 'Date','Major','Class','Mid Term Grade','Department','Subject','Course','ID','Academic Period','Major College']] #Reordering the Columns
hubcontacts = hubcontacts.sort_values(by = "MT Intervention Group") #Sorting the table to have newer values first
hubcontacts

,MT Intervention Group,Notes,Advising Pin,Hold,Advisor,Date,Major,Class,Mid Term Grade,Department,Subject,Course,ID,Academic Period,Major College
0,1,NaN,NaN,NaN,NaN,NaN,ID,FR,C-,ARCH,AED,22860,33,202280,CAED
1,1,NaN,NaN,NaN,NaN,NaN,ID,SO,D,ARCH,AED,22860,31,202280,CAED
2,1,NaN,NaN,NaN,NaN,NaN,ID,FR,C,ARCH,AED,22860,11,202280,CAED
3,1,NaN,NaN,NaN,NaN,NaN,ID,FR,F,ARCH,AED,22860,78,202280,CAED
4,1,NaN,NaN,NaN,NaN,NaN,ID,FR,C,ARCH,AED,22860,69,202280,CAED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3592,2,NaN,NaN,NaN,NaN,NaN,FD,JR,SF,FDM,FDM,20907,49367,202480,CotA
5776,2,NaN,NaN,NaN,NaN,NaN,FD,JR,F,MDJ,MDJ,15027,62603,202610,CotA
5489,2,NaN,NaN,NaN,NaN,NaN,FD,JR,F,ART,ARTH,17400,15778,202580,CotA
5490,2,NaN,NaN,NaN,NaN,NaN,ARTE,SR,F,ART,ARTH,17400,15854,202580,CotA


##### <center>*Creating College-Specific Dataframes*</center>

In [497]:
#Section 6e - Creating College-Specific Dataframes
caedcontacts = hubcontacts[(hubcontacts["Major College"] == "CAED") & (hubcontacts["Academic Period"].isin(currentterm))]
ccicontacts = hubcontacts[(hubcontacts["Major College"] == "CCI") & (hubcontacts["Academic Period"].isin(currentterm))]
cotacontacts = hubcontacts[(hubcontacts["Major College"] == "CotA") & (hubcontacts["Academic Period"].isin(currentterm))]

##### <center>*Creating the Intervention Contact Excel File*</center>

The Excel file will be posted to a single file. I think that having a single file is preferable to multiple files - that way the advising director in the hub can review the units in a single place, and can split it up further if needed (i.e. if one unit has a heavier workload, that could be split among other units to lighten the load)

In [498]:
#Section 6f - Creating the Excel file and Tabs
interwriter = pd.ExcelWriter(f"{prettyterm} Advisor Intervention Contact List.xlsx",
                             engine = "xlsxwriter")

caedcontacts.to_excel(interwriter, index = False, sheet_name = "CAEDInterventions")
ccicontacts.to_excel(interwriter, index = False, sheet_name = "CCIInterventions")
cotacontacts.to_excel(interwriter, index = False, sheet_name = "CotAInterventions")

In [499]:
#Section 6g - Defining the variables for editing the book and unique sheets
interbook = interwriter.book
caedintersheet = interwriter.sheets["CAEDInterventions"]
cciintersheet = interwriter.sheets["CCIInterventions"]
cotaintersheet = interwriter.sheets["CotAInterventions"]

In [500]:
#Section 6h - Establishing formatting for the File
#I'm trying to keep with the style guide as touched on before
header_format = interbook.add_format({
    "font_name": "Andale WT",
    "font_size": 8,
    "bold": True,
    "font_color":"#FFFFFF",
    "bg_color": "#003976",
    "align":"center"
})

caedcolumns = [{'header': column, "header_format":header_format} for column in caedcontacts.columns]
ccicolumns = [{'header': column, "header_format":header_format} for column in ccicontacts.columns]
cotacolumns = [{'header': column, "header_format":header_format} for column in cotacontacts.columns]

#This is a format for the columns related to the student info (major, grade, etc.). This is an artifact from the older files, and I am partial to removing this for future iterations of the file.
#It doesn't necessarilly add anything - adding color to the group column may be better, but I can see the merit (highlights majors, grades, etc. that advisors may want to hone in on prior to reaching out)
alertformat = interbook.add_format({"bg_color":'#FFCCCC',
                                    "font_name": "Andale WT",
                                    "font_size": 8,
                                    "align": "center"
})

rowcustoms = interbook.add_format({
    "font_name": "Andale WT",
    "font_size": 8,
    "align": "center"
})

In [501]:
#Section 6i - Formatting the CAED Tab
(caedrowmax, caedcolmax) = caedcontacts.shape #This sets the shape of the file - this allows us to control what row(s) are being manipulated

caedintersheet.add_table(0, 0, caedrowmax, caedcolmax - 1, {
    "columns": caedcolumns,
    "style": "Table Style Medium 2" #I looked over the table styles and this one meshes the best with the #003976 color
})

caedintersheet.conditional_format(1, 6, caedrowmax, 8, { #This specifies the columns that are going to be impacted by the alertformat
    "type": "no_errors",
    "format": alertformat
})

0

In [502]:
#Section 6j - Formatting the CCI Tab
(ccirowmax, ccicolmax) = ccicontacts.shape 

cciintersheet.add_table(0, 0, ccirowmax, ccicolmax - 1, {
    "columns": ccicolumns,
    "style": "Table Style Medium 2" 
})

cciintersheet.conditional_format(1, 6, ccirowmax, 8, {
    "type": "no_errors", 
    "format": alertformat
})

0

In [503]:
#Section 6k - Formatting the CotA Tab
(cotarowmax, cotacolmax) = cotacontacts.shape

cotaintersheet.add_table(0, 0, cotarowmax, cotacolmax - 1, {
    "columns": cotacolumns,
    "style": "Table Style Medium 2" 
})

cotaintersheet.conditional_format(1, 6, cotarowmax, 8, {
    "type": "no_errors",
    "format": alertformat
})

0

In [504]:
#Section 6l - #Setting the Column Sizing & Closing the Book
caedintersheet.set_column(0, caedcolmax-1, 20, rowcustoms) #Opted to set the rows to be 20 pixels wide - gives enough to see the full data, but not overwhelming
caedintersheet.set_column(1, 1, 80, rowcustoms) #This expands the Notes column - that gives more space for advisors (I worry if it was smaller, it'd discourage detailed notes)
caedintersheet.freeze_panes(1,0) #This allows for the headers to remain visible at all time - key for scrolling through the file!

cciintersheet.set_column(0, ccicolmax-1, 20, rowcustoms)
cciintersheet.set_column(1, 1, 80, rowcustoms)
cciintersheet.freeze_panes(1,0)

cotaintersheet.set_column(0, cotacolmax-1, 20, rowcustoms)
cotaintersheet.set_column(1, 1, 80, rowcustoms)
cotaintersheet.freeze_panes(1,0)

interwriter.close()

#### <center>**Section 7 - Midterm-to-Final Grade Matrix**</center>

The code in this section is dedicated to create heat maps for departments (CCI and CotA) or subjects (CAED) to highlight grade changes from Midterm grades to Final grades. Special attention is given to the students who received interventions - the section with students eligible for interventions are encircled by an orange square.

This code will be utilizing the **mthubonlyclean.csv** file that was made previously. This way, only students within the hub will be reviewed. The thought process is that this minimizes the pool of students being examined, as well as compares students of like majors (i.e. only looking at students within the hub, not students outside who may struggle in content-specific courses).

##### <center>*Reloading the Dataframe*</center>

In [505]:
#Section 7a - Reloading the Data
mtheat = pd.read_csv("mthubonlyclean.csv")
mtheat.shape

(58787, 18)

##### <center>*Removing Term w/o Final Grades*</center>

The most recent term w/o final grades is dictated by the **currentheatterm** variable found in Section 0b.

In [506]:
#Section 7b - Removing most recent term w/o Final Grades
#Since the most recent term doesn't have final grades yet, this can remove that term so it doesn't cause errors (doesn't have a final grade to compare too)
mtheat = mtheat.loc[~mtheat["Academic Period"].isin(currentheatterm)]
mtheat.shape

(52328, 18)

##### <center>*Mapping the Terms*</center>

Like always, we are mapping the terms to the full title. The difference is we are flipping the terms - we want the newest term first so that can be the "default" state of the visual

In [507]:
mtheat = mtheat.sort_values("Academic Period", ascending = False)
mtheat["Academic Period"].unique()

array([202580, 202510, 202480, 202410, 202380, 202310, 202280])

In [508]:
#Section 7b - Mapping the Terms
mtheat["Academic Period"] = mtheat["Academic Period"].map(figtermmap)
mtheat["Academic Period"].unique()

<StringArray>
[  'Fall 2025', 'Spring 2025',   'Fall 2024', 'Spring 2024',   'Fall 2023',
 'Spring 2023',   'Fall 2022']
Length: 7, dtype: str

In [509]:
#Section 7c - Creating the Term List
heattermlist = mtheat["Academic Period"].unique()

##### <center>*Creating Subject/Department Dataframes*</center>

In [510]:
#Section 7c - Creating Unit-Specific Dataframes
aedheat = mtheat[mtheat["Subject"] == "AED"]
archheat = mtheat[mtheat["Subject"] == "ARCH"]
arcsheat = mtheat[mtheat["Subject"] == "ARCS"]
cmgtheat = mtheat[mtheat["Subject"] == "CMGT"]
idheat = mtheat[mtheat["Subject"] == "ID"]
cciheat = mtheat[mtheat["Department"] == "CCI"]
commheat = mtheat[mtheat["Department"] == "COMM"]
ematheat = mtheat[mtheat["Department"] == "EMAT"]
mdjheat = mtheat[mtheat["Department"] == "MDJ"]
vcdheat = mtheat[mtheat["Department"] == "VCD"]
artheat = mtheat[mtheat["Subject"] == "ART"]
arthheat = mtheat[mtheat["Subject"]  == "ARTH"]
artsheat = mtheat[mtheat["Subject"]  == "ARTS"]
fdmheat = mtheat[mtheat["Department"] == "FDM"]
musheat = mtheat[mtheat["Department"] == "MUS"]
danheat = mtheat[mtheat["Subject"] == "DAN"]
theaheat = mtheat[mtheat["Subject"]  == "THEA"]

##### <center>*Creating the Heat Maps*</center>

It is important to note that the Heat Map underwent many iterations between now and the "final" version below. This is due in part to the need to import the visual from VS Code to PowerBI. You can read more about the changes in the "mtheatmapfails.ipynb" file, but the short version is *PowerBI could not read the texttemplate function, so I had to revert to a more traditional Heat Map*. The planning and attempt to rebuild the visualizations was done with Google Gemini - this is also detailed in the previously mentioned file.

In [511]:
#Section 7d - AED Heat Map
buttons = [] #Creates the buttons array
zmatrix = {} #This is creating the data for the z value - this is what we will be putting in for the heat map

initial_term = heattermlist[0] #This is setting the initial term for the figure - this helps create the default state for the visual by calling the most recent term

for term in heattermlist:
    termframe = aedheat[aedheat["Academic Period"] == term] #Calling the terms

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"]) #This is what creates the "meat" of the figure - it counts every instance where there is overlap of the final and midterm grades
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0) #This helps format the crosstab above, while also filling in 0 values. Without it, it can throw the figure off since it will skip N/A values

    zmatrix[term] = termcrosstab.values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) #This allows the maximum value to fluctuate depending on the zmatrix
    zbot = min(min(row) for row in zmatrix[term]) #This allows the minimum value to flucuate depending on the zmatrx

    buttons.append(dict( #Setting the buttons up
        method = "update",
        label = term,
        args = [{ #these are what update every term
            "z": [zmatrix[term]], #changes the z value to the zmatrix for the chosen term
            "zmin": [zbot], #updates the lowest value
            "zmax": [ztop], #updates the highest value
        }]
    ))



recentheattrace = go.Heatmap(#For the most recent term
    z = zmatrix[initial_term], #This part sets the cross tab up to show the accurate data
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, #calling the earlier variable, this helps set the x and y axes to be in the appropriate order
    y = gradescalestring,
    xgap = 1, #This helps create a little gap between the cells - this helps further define the cells to pop more and clearly define the sectors
    ygap = 1,
    hovertemplate = (#Since we have x, y and z - we can call the z value instead of adding custom data
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom, #I had to tweak the Cividis color range - this way lower values actually show up when dwarfed by larger values (typically 4.0 x 4.0). It also goes well with the blue/gold scheme, and colorblind friendly!
    colorbar = dict(
        title = dict( text = "<b>Student Density</b>", #I felt it is important to understand that this chart is density
        font_color = "black")
        ),
    visible = True)

pastheattrace = go.Heatmap(#For the older terms
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text= "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = False)

legendtrace = go.Scatter( #So I wanted to highlight the students that may get interventions - this is formally set up later in the annotations, but this creates a blank trace with a legend!
    x = [None], #doesn't show up
    y = [None], #doesn't show up
    mode = "lines",
    line = dict( #this helps align with the shape below and make it clear they are the same thing
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "<b>AED Grade Movement from Midterms to Final</b>",
        font_size = 24,
        font_color = "black",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .96),
    plot_bgcolor = "black",
    xaxis_showgrid = False,
    yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 550,
    width = 730,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .575,
        y = 1.135
    )],
    legend = dict(
        orientation = "h",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = -.15),
    font_family = "Segoe UI"
    )

aedheat = go.Figure(data = [recentheattrace, pastheattrace, legendtrace], layout = layout)

aedheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3925,
    y = 1.1275,
    xref = "paper",
    yref = "paper",
    font_color = "black",
    font_size = 16
)
aedheat.add_shape(#This is the shape that highlights the interventions received
    type = "rect", 
    line = dict(
        color = "#FF8C00", #I felt orange was bright enough to draw attention, but not bright enough that it was distracting. It also calls to mind a warning
        width = 2, #The width somehow fits perfectly between the gap, so I opted for that - this way it doesn't overtake the figure
        dash = "dash" #I thought about doing a solid line, but opted for the dash. I felt that it looks nicer and brings attention to it.
    ),
    x0 = -0.5, #These coordinates have it hone in right on the section covered. I had to do some logicing out with it, but after realizing that .5 put it in between indices, it worked!
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

aedheat.show()

In [512]:
#Section 7e - ARCH Heat Map
buttons = []
zmatrix = {}

initial_term = heattermlist[0]

for term in heattermlist:
    termframe = archheat[archheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0)

    zmatrix[term] = termcrosstab.values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) 
    zbot = min(min(row) for row in zmatrix[term])

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{ #these are what update every term
            "z": [zmatrix[term]], 
            "zmin": [zbot],
            "zmax": [ztop],
        }]
    ))



recentheattrace = go.Heatmap(
    z = zmatrix[initial_term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text = "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = True)

pastheattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text= "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = False)

legendtrace = go.Scatter(
    x = [None],
    y = [None],
    mode = "lines",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "<b>ARCH Grade Movement from Midterms to Final</b>",
        font_size = 24,
        font_color = "black",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .96),
    plot_bgcolor = "black",
    xaxis_showgrid = False,
    yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 550,
    width = 730,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .575,
        y = 1.135
    )],
    legend = dict(
        orientation = "h",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = -.15),
    font_family = "Segoe UI"
    )

archheat = go.Figure(data = [recentheattrace, pastheattrace, legendtrace], layout = layout)

archheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3925,
    y = 1.1275,
    xref = "paper",
    yref = "paper",
    font_color = "black",
    font_size = 16
)
archheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

archheat.show()

In [513]:
#Section 7f - ARCS Heat Map
buttons = []
zmatrix = {}

initial_term = heattermlist[0]

for term in heattermlist:
    termframe = arcsheat[arcsheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0)

    zmatrix[term] = termcrosstab.values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) 
    zbot = min(min(row) for row in zmatrix[term])

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{ #these are what update every term
            "z": [zmatrix[term]], 
            "zmin": [zbot],
            "zmax": [ztop],
        }]
    ))



recentheattrace = go.Heatmap(
    z = zmatrix[initial_term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text = "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = True)

pastheattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text= "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = False)

legendtrace = go.Scatter(
    x = [None],
    y = [None],
    mode = "lines",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "<b>ARCS Grade Movement from Midterms to Final</b>",
        font_size = 24,
        font_color = "black",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .96),
    plot_bgcolor = "black",
    xaxis_showgrid = False,
    yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 550,
    width = 730,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .575,
        y = 1.135
    )],
    legend = dict(
        orientation = "h",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = -.15),
    font_family = "Segoe UI"
    )

arcsheat = go.Figure(data = [recentheattrace, pastheattrace, legendtrace], layout = layout)

arcsheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3925,
    y = 1.1275,
    xref = "paper",
    yref = "paper",
    font_color = "black",
    font_size = 16
)
arcsheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

arcsheat.show()

In [514]:
#Section 7g - CMGT Heat Map
buttons = []
zmatrix = {}

initial_term = heattermlist[0]

for term in heattermlist:
    termframe = cmgtheat[cmgtheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0)

    zmatrix[term] = termcrosstab.values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) 
    zbot = min(min(row) for row in zmatrix[term])

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{ #these are what update every term
            "z": [zmatrix[term]], 
            "zmin": [zbot],
            "zmax": [ztop],
        }]
    ))



recentheattrace = go.Heatmap(
    z = zmatrix[initial_term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text = "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = True)

pastheattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text= "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = False)

legendtrace = go.Scatter(
    x = [None],
    y = [None],
    mode = "lines",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "<b>CMGT Grade Movement from Midterms to Final</b>",
        font_size = 24,
        font_color = "black",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .96),
    plot_bgcolor = "black",
    xaxis_showgrid = False,
    yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 550,
    width = 730,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .575,
        y = 1.135
    )],
    legend = dict(
        orientation = "h",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = -.15),
    font_family = "Segoe UI"
    )

cmgtheat = go.Figure(data = [recentheattrace, pastheattrace, legendtrace], layout = layout)

cmgtheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3925,
    y = 1.1275,
    xref = "paper",
    yref = "paper",
    font_color = "black",
    font_size = 16
)
cmgtheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

cmgtheat.show()

In [515]:
#Section 7h - ID Heat Map
buttons = []
zmatrix = {}

initial_term = heattermlist[0]

for term in heattermlist:
    termframe = idheat[idheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0)

    zmatrix[term] = termcrosstab.values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) 
    zbot = min(min(row) for row in zmatrix[term])

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{ #these are what update every term
            "z": [zmatrix[term]], 
            "zmin": [zbot],
            "zmax": [ztop],
        }]
    ))



recentheattrace = go.Heatmap(
    z = zmatrix[initial_term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text = "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = True)

pastheattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text= "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = False)

legendtrace = go.Scatter(
    x = [None],
    y = [None],
    mode = "lines",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "<b>ID Grade Movement from Midterms to Final</b>",
        font_size = 24,
        font_color = "black",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .96),
    plot_bgcolor = "black",
    xaxis_showgrid = False,
    yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 550,
    width = 730,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .575,
        y = 1.135
    )],
    legend = dict(
        orientation = "h",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = -.15),
    font_family = "Segoe UI"
    )

idheat = go.Figure(data = [recentheattrace, pastheattrace, legendtrace], layout = layout)

idheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3925,
    y = 1.1275,
    xref = "paper",
    yref = "paper",
    font_color = "black",
    font_size = 16
)
idheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

idheat.show()

In [516]:
#Section 7i - CCI Heat Map
buttons = []
zmatrix = {}

initial_term = heattermlist[0]

for term in heattermlist:
    termframe = cciheat[cciheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0)

    zmatrix[term] = termcrosstab.values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) 
    zbot = min(min(row) for row in zmatrix[term])

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{ #these are what update every term
            "z": [zmatrix[term]], 
            "zmin": [zbot],
            "zmax": [ztop],
        }]
    ))



recentheattrace = go.Heatmap(
    z = zmatrix[initial_term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text = "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = True)

pastheattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text= "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = False)

legendtrace = go.Scatter(
    x = [None],
    y = [None],
    mode = "lines",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "<b>CCI Grade Movement from Midterms to Final</b>",
        font_size = 24,
        font_color = "black",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .96),
    plot_bgcolor = "black",
    xaxis_showgrid = False,
    yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 550,
    width = 730,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .575,
        y = 1.135
    )],
    legend = dict(
        orientation = "h",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = -.15),
    font_family = "Segoe UI"
    )

cciheat = go.Figure(data = [recentheattrace, pastheattrace, legendtrace], layout = layout)

cciheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3925,
    y = 1.1275,
    xref = "paper",
    yref = "paper",
    font_color = "black",
    font_size = 16
)
cciheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

cciheat.show()

In [517]:
#Section 7j - COMM Heat Map
buttons = []
zmatrix = {}

initial_term = heattermlist[0]

for term in heattermlist:
    termframe = commheat[commheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0)

    zmatrix[term] = termcrosstab.values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) 
    zbot = min(min(row) for row in zmatrix[term])

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{ #these are what update every term
            "z": [zmatrix[term]], 
            "zmin": [zbot],
            "zmax": [ztop],
        }]
    ))



recentheattrace = go.Heatmap(
    z = zmatrix[initial_term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text = "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = True)

pastheattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text= "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = False)

legendtrace = go.Scatter(
    x = [None],
    y = [None],
    mode = "lines",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "<b>COMM Grade Movement from Midterms to Final</b>",
        font_size = 24,
        font_color = "black",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .96),
    plot_bgcolor = "black",
    xaxis_showgrid = False,
    yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 550,
    width = 730,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .575,
        y = 1.135
    )],
    legend = dict(
        orientation = "h",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = -.15),
    font_family = "Segoe UI"
    )

commheat = go.Figure(data = [recentheattrace, pastheattrace, legendtrace], layout = layout)

commheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3925,
    y = 1.1275,
    xref = "paper",
    yref = "paper",
    font_color = "black",
    font_size = 16
)
commheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

commheat.show()

In [518]:
#Section 7k - EMAT Heat Map
buttons = []
zmatrix = {}

initial_term = heattermlist[0]

for term in heattermlist:
    termframe = ematheat[ematheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0)

    zmatrix[term] = termcrosstab.values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) 
    zbot = min(min(row) for row in zmatrix[term])

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{ #these are what update every term
            "z": [zmatrix[term]], 
            "zmin": [zbot],
            "zmax": [ztop],
        }]
    ))



recentheattrace = go.Heatmap(
    z = zmatrix[initial_term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text = "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = True)

pastheattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text= "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = False)

legendtrace = go.Scatter(
    x = [None],
    y = [None],
    mode = "lines",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "<b>EMAT Grade Movement from Midterms to Final</b>",
        font_size = 24,
        font_color = "black",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .96),
    plot_bgcolor = "black",
    xaxis_showgrid = False,
    yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 550,
    width = 730,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .575,
        y = 1.135
    )],
    legend = dict(
        orientation = "h",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = -.15),
    font_family = "Segoe UI"
    )

ematheat = go.Figure(data = [recentheattrace, pastheattrace, legendtrace], layout = layout)

ematheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3925,
    y = 1.1275,
    xref = "paper",
    yref = "paper",
    font_color = "black",
    font_size = 16
)
ematheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

ematheat.show()

In [519]:
#Section 7l - MDJ Heat Map
buttons = []
zmatrix = {}

initial_term = heattermlist[0]

for term in heattermlist:
    termframe = mdjheat[mdjheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0)

    zmatrix[term] = termcrosstab.values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) 
    zbot = min(min(row) for row in zmatrix[term])

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{ #these are what update every term
            "z": [zmatrix[term]], 
            "zmin": [zbot],
            "zmax": [ztop],
        }]
    ))



recentheattrace = go.Heatmap(
    z = zmatrix[initial_term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text = "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = True)

pastheattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text= "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = False)

legendtrace = go.Scatter(
    x = [None],
    y = [None],
    mode = "lines",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "<b>MDJ Grade Movement from Midterms to Final</b>",
        font_size = 24,
        font_color = "black",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .96),
    plot_bgcolor = "black",
    xaxis_showgrid = False,
    yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 550,
    width = 730,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .575,
        y = 1.135
    )],
    legend = dict(
        orientation = "h",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = -.15),
    font_family = "Segoe UI"
    )

mdjheat = go.Figure(data = [recentheattrace, pastheattrace, legendtrace], layout = layout)

mdjheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3925,
    y = 1.1275,
    xref = "paper",
    yref = "paper",
    font_color = "black",
    font_size = 16
)
mdjheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

mdjheat.show()

In [520]:
#Section 7m - VCD Heat Map
buttons = []
zmatrix = {}

initial_term = heattermlist[0]

for term in heattermlist:
    termframe = vcdheat[vcdheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0)

    zmatrix[term] = termcrosstab.values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) 
    zbot = min(min(row) for row in zmatrix[term])

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{ #these are what update every term
            "z": [zmatrix[term]], 
            "zmin": [zbot],
            "zmax": [ztop],
        }]
    ))



recentheattrace = go.Heatmap(
    z = zmatrix[initial_term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text = "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = True)

pastheattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text= "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = False)

legendtrace = go.Scatter(
    x = [None],
    y = [None],
    mode = "lines",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "<b>VCD Grade Movement from Midterms to Final</b>",
        font_size = 24,
        font_color = "black",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .96),
    plot_bgcolor = "black",
    xaxis_showgrid = False,
    yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 550,
    width = 730,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .575,
        y = 1.135
    )],
    legend = dict(
        orientation = "h",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = -.15),
    font_family = "Segoe UI"
    )

vcdheat = go.Figure(data = [recentheattrace, pastheattrace, legendtrace], layout = layout)

vcdheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3925,
    y = 1.1275,
    xref = "paper",
    yref = "paper",
    font_color = "black",
    font_size = 16
)
vcdheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

vcdheat.show()

In [521]:
#Section 7n - ART Heat Map
buttons = []
zmatrix = {}

initial_term = heattermlist[0]

for term in heattermlist:
    termframe = artheat[artheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0)

    zmatrix[term] = termcrosstab.values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) 
    zbot = min(min(row) for row in zmatrix[term])

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{ #these are what update every term
            "z": [zmatrix[term]], 
            "zmin": [zbot],
            "zmax": [ztop],
        }]
    ))



recentheattrace = go.Heatmap(
    z = zmatrix[initial_term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text = "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = True)

pastheattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text= "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = False)

legendtrace = go.Scatter(
    x = [None],
    y = [None],
    mode = "lines",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "<b>ART Grade Movement from Midterms to Final</b>",
        font_size = 24,
        font_color = "black",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .96),
    plot_bgcolor = "black",
    xaxis_showgrid = False,
    yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 550,
    width = 730,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .575,
        y = 1.135
    )],
    legend = dict(
        orientation = "h",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = -.15),
    font_family = "Segoe UI"
    )

artheat = go.Figure(data = [recentheattrace, pastheattrace, legendtrace], layout = layout)

artheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3925,
    y = 1.1275,
    xref = "paper",
    yref = "paper",
    font_color = "black",
    font_size = 16
)
artheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

artheat.show()

In [522]:
#Section 7n(b) - ARTH Heat Map
buttons = []
zmatrix = {}

initial_term = heattermlist[0]

for term in heattermlist:
    termframe = arthheat[arthheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0)

    zmatrix[term] = termcrosstab.values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) 
    zbot = min(min(row) for row in zmatrix[term])

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{ #these are what update every term
            "z": [zmatrix[term]], 
            "zmin": [zbot],
            "zmax": [ztop],
        }]
    ))



recentheattrace = go.Heatmap(
    z = zmatrix[initial_term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text = "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = True)

pastheattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text= "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = False)

legendtrace = go.Scatter(
    x = [None],
    y = [None],
    mode = "lines",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "<b>ARTH Grade Movement from Midterms to Final</b>",
        font_size = 24,
        font_color = "black",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .96),
    plot_bgcolor = "black",
    xaxis_showgrid = False,
    yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 550,
    width = 730,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .575,
        y = 1.135
    )],
    legend = dict(
        orientation = "h",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = -.15),
    font_family = "Segoe UI"
    )

arthheat = go.Figure(data = [recentheattrace, pastheattrace, legendtrace], layout = layout)

arthheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3925,
    y = 1.1275,
    xref = "paper",
    yref = "paper",
    font_color = "black",
    font_size = 16
)
arthheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

arthheat.show()

In [523]:
#Section 7n(c) - ARTS Heat Map
buttons = []
zmatrix = {}

initial_term = heattermlist[0]

for term in heattermlist:
    termframe = artsheat[artsheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0)

    zmatrix[term] = termcrosstab.values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) 
    zbot = min(min(row) for row in zmatrix[term])

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{ #these are what update every term
            "z": [zmatrix[term]], 
            "zmin": [zbot],
            "zmax": [ztop],
        }]
    ))



recentheattrace = go.Heatmap(
    z = zmatrix[initial_term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text = "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = True)

pastheattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text= "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = False)

legendtrace = go.Scatter(
    x = [None],
    y = [None],
    mode = "lines",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "<b>ARTS Grade Movement from Midterms to Final</b>",
        font_size = 24,
        font_color = "black",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .96),
    plot_bgcolor = "black",
    xaxis_showgrid = False,
    yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 550,
    width = 730,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .575,
        y = 1.135
    )],
    legend = dict(
        orientation = "h",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = -.15),
    font_family = "Segoe UI"
    )

artsheat = go.Figure(data = [recentheattrace, pastheattrace, legendtrace], layout = layout)

artsheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3925,
    y = 1.1275,
    xref = "paper",
    yref = "paper",
    font_color = "black",
    font_size = 16
)
artsheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

artsheat.show()

In [524]:
#Section 7p - MUS Heat Map
buttons = []
zmatrix = {}

initial_term = heattermlist[0]

for term in heattermlist:
    termframe = musheat[musheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0)

    zmatrix[term] = termcrosstab.values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) 
    zbot = min(min(row) for row in zmatrix[term])

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{ #these are what update every term
            "z": [zmatrix[term]], 
            "zmin": [zbot],
            "zmax": [ztop],
        }]
    ))



recentheattrace = go.Heatmap(
    z = zmatrix[initial_term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text = "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = True)

pastheattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text= "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = False)

legendtrace = go.Scatter(
    x = [None],
    y = [None],
    mode = "lines",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "<b>MUS Grade Movement from Midterms to Final</b>",
        font_size = 24,
        font_color = "black",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .96),
    plot_bgcolor = "black",
    xaxis_showgrid = False,
    yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 550,
    width = 730,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .575,
        y = 1.135
    )],
    legend = dict(
        orientation = "h",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = -.15),
    font_family = "Segoe UI"
    )

musheat = go.Figure(data = [recentheattrace, pastheattrace, legendtrace], layout = layout)

musheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3925,
    y = 1.1275,
    xref = "paper",
    yref = "paper",
    font_color = "black",
    font_size = 16
)
musheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

musheat.show()

In [525]:
#Section 7q - FDM Heat Map
buttons = []
zmatrix = {}

initial_term = heattermlist[0]

for term in heattermlist:
    termframe = fdmheat[fdmheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0)

    zmatrix[term] = termcrosstab.values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) 
    zbot = min(min(row) for row in zmatrix[term])

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{ #these are what update every term
            "z": [zmatrix[term]], 
            "zmin": [zbot],
            "zmax": [ztop],
        }]
    ))



recentheattrace = go.Heatmap(
    z = zmatrix[initial_term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text = "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = True)

pastheattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text= "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = False)

legendtrace = go.Scatter(
    x = [None],
    y = [None],
    mode = "lines",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "<b>FDM Grade Movement from Midterms to Final</b>",
        font_size = 24,
        font_color = "black",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .96),
    plot_bgcolor = "black",
    xaxis_showgrid = False,
    yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 550,
    width = 730,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .575,
        y = 1.135
    )],
    legend = dict(
        orientation = "h",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = -.15),
    font_family = "Segoe UI"
    )

fdmheat = go.Figure(data = [recentheattrace, pastheattrace, legendtrace], layout = layout)

fdmheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3925,
    y = 1.1275,
    xref = "paper",
    yref = "paper",
    font_color = "black",
    font_size = 16
)
fdmheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

fdmheat.show()

In [526]:
#Section 7r - DAN Heat Map
buttons = []
zmatrix = {}

initial_term = heattermlist[0]

for term in heattermlist:
    termframe = danheat[danheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0)

    zmatrix[term] = termcrosstab.values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) 
    zbot = min(min(row) for row in zmatrix[term])

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{ #these are what update every term
            "z": [zmatrix[term]], 
            "zmin": [zbot],
            "zmax": [ztop],
        }]
    ))



recentheattrace = go.Heatmap(
    z = zmatrix[initial_term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text = "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = True)

pastheattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text= "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = False)

legendtrace = go.Scatter(
    x = [None],
    y = [None],
    mode = "lines",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "<b>DAN Grade Movement from Midterms to Final</b>",
        font_size = 24,
        font_color = "black",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .96),
    plot_bgcolor = "black",
    xaxis_showgrid = False,
    yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 550,
    width = 730,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .575,
        y = 1.135
    )],
    legend = dict(
        orientation = "h",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = -.15),
    font_family = "Segoe UI"
    )

danheat = go.Figure(data = [recentheattrace, pastheattrace, legendtrace], layout = layout)

danheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3925,
    y = 1.1275,
    xref = "paper",
    yref = "paper",
    font_color = "black",
    font_size = 16
)
danheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

danheat.show()

In [527]:
#Section 7s - THEA Heat Map
buttons = []
zmatrix = {}

initial_term = heattermlist[0]

for term in heattermlist:
    termframe = theaheat[theaheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0)

    zmatrix[term] = termcrosstab.values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) 
    zbot = min(min(row) for row in zmatrix[term])

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{ #these are what update every term
            "z": [zmatrix[term]], 
            "zmin": [zbot],
            "zmax": [ztop],
        }]
    ))



recentheattrace = go.Heatmap(
    z = zmatrix[initial_term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text = "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = True)

pastheattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        title = dict( text= "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = False)

legendtrace = go.Scatter(
    x = [None],
    y = [None],
    mode = "lines",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "<b>THEA Grade Movement from Midterms to Final</b>",
        font_size = 24,
        font_color = "black",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .96),
    plot_bgcolor = "black",
    xaxis_showgrid = False,
    yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 550,
    width = 730,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .575,
        y = 1.135
    )],
    legend = dict(
        orientation = "h",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = -.15),
    font_family = "Segoe UI"
    )

theaheat = go.Figure(data = [recentheattrace, pastheattrace, legendtrace], layout = layout)

theaheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3925,
    y = 1.1275,
    xref = "paper",
    yref = "paper",
    font_color = "black",
    font_size = 16
)
theaheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

theaheat.show()

#### <center>*Section 8 - GPA Conversion Chart*</center>

In order to make the project in Python as much as possible, I am going to create a small chart that shows the grade conversion rates to be featured alongside the Heat Map. It is essentially converting the table found in Section 1 into a graph object.

This was a last minute addition, but I attempted to follow the design logic of the previous tables (alternating rows, colors, etc.).

In [528]:
#Section 8a - Creating the GPA Dataframe
gpachartdf = {
    "Letter Grade" : ["A","A-","B+","B","B-","C+","C","C-","D+","D","F","S","U","W","SF","NF","AU","NaN","DR","IN","NR"],
    "GPA Conversion": [4.0, 3.7, 3.3, 3.0, 2.7, 2.3, 2.0, 1.7, 1.3, 1.0, 0.0, 4.0, 0.0 , 0.0, 0.0, "NaN","NaN","NaN","NaN","NaN","NaN"],
    "Intervention Eligible?" : ["Passing Grade - No Intervention Needed", 
                                "Passing Grade - No Intervention Needed", 
                                "Passing Grade - No Intervention Needed", 
                                "Passing Grade - No Intervention Needed", 
                                "Passing Grade - No Intervention Needed", 
                                "Passing Grade - No Intervention Needed", 
                                "Intervention Required (FR/SO Only)", 
                                "Intervention Required (FR/SO Only)", 
                                "Intervention Required (FR/SO Only)", 
                                "Intervention Required (FR/SO Only)", 
                                "Intervention Required", 
                                "Passing Grade - No Intervention Needed", 
                                "Intervention Required", 
                                "No Intervention (Withdrawn)", #I wanted to still highlight that withdraws are 0.0, but do not get an intervention - this is important since they count for DFW rates
                                "Intervention Required", 
                                "No Grade Reported", 
                                "No Grade Reported", 
                                "No Grade Reported", 
                                "No Grade Reported", 
                                "No Grade Reported", 
                                "No Grade Reported"]
    }

gpachart = pd.DataFrame(gpachartdf)
gpachart

,Letter Grade,GPA Conversion,Intervention Eligible?
0,A,4.0,Passing Grade - No Intervention Needed
1,A-,3.7,Passing Grade - No Intervention Needed
2,B+,3.3,Passing Grade - No Intervention Needed
3,B,3.0,Passing Grade - No Intervention Needed
4,B-,2.7,Passing Grade - No Intervention Needed
5,C+,2.3,Passing Grade - No Intervention Needed
6,C,2.0,Intervention Required (FR/SO Only)
7,C-,1.7,Intervention Required (FR/SO Only)
8,D+,1.3,Intervention Required (FR/SO Only)
9,D,1.0,Intervention Required (FR/SO Only)


In [529]:
#Section 8b - Creating Conditional Formatting
gparows = len(gpachart)
gpazebra = ["#FFFFFF", "#FFF1CC"] * (gparows // 2+1)
gpazebra = gpazebra[:gparows]

intercriteria = []
for val in gpachart["Intervention Eligible?"]:
    if val == "Passing Grade - No Intervention Needed":
        intercriteria.append("#CCFFCC")
    elif val == "Intervention Required (FR/SO Only)":
        intercriteria.append("#FFFFCC")
    elif val == "Intervention Required":
        intercriteria.append("#FFCCCC")
    else:
        intercriteria.append("#B1B1B1")

In [530]:
#Section 8c - Creating the Table
#Most of the design philosophy adheres to the other tables
gpatrace = go.Table(
        columnwidth = [1,1,4], #This helps size out the table so the third column (with the most text) can be shown in full
        header = dict(values = ["Letter Grade", "GPA Conversion", "Intervention Needed"],
                      line_color = "black",
                      fill_color = "#EFAB00",
                      font_color = "black",
                      font_weight = "bold"),
        cells = dict(values = [gpachart["Letter Grade"], gpachart["GPA Conversion"], gpachart["Intervention Eligible?"]],
                     line_color = "black",
                     fill_color = [gpazebra, gpazebra, intercriteria],
                     font_color = "black")
)

gpalayout = go.Layout(
        title = dict(text = f"<b>GPA Conversion Chart</b>",
                 xanchor = "center",
                 x = .5,
                 font_color = "black",
                 font_size = 20),
        font_family = "Segoe UI",
        margin = dict(l=20, r=20, b=20, t = 50),
        width = 450,
        height = 540
)

gpatab = go.Figure(data = gpatrace, layout = gpalayout)

gpatab.show()

#### <center>**Section 9 - HTML Conversion Code**</center>

The code in this section is focused on converting the figures made above (Sections 2-5 and 7) into JSON files embedded in HTML code. This is what will allow the posting of the figures on the PowerBI file for presentation. Once all figures have been converted into the code strings, they will be converted into an Excel file. This file is what will be read by PowerBI, as opposed to copying and pasting the HTML code into PowerBI.

*Note on the code below*: The general formatting for the code below was made in conjunction with Google Gemini. This ensured smooth translation into PowerBI's HTML visual. It is also important to note small tweaks to the figures above (i.e. adding tolist to the line charts), were done post-creation under the guidance of Gemini to get the figures to load correctly.

##### <center>*Midterm Grade Summary Tables HTML Conversion*</center>

In [531]:
#CAED Summary Table HTML Code
caedsumtabjson = caedsumtable.to_json()

caedsumtabhtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {caedsumtabjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(caedsumtabhtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"cells":{"fill":{"color":[["#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF"],["#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF"],["#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF"],["#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF"],[

In [532]:
#CCI Summary Table HTML Code
ccisumtabjson = ccisumtable.to_json()

ccisumtabhtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {ccisumtabjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(ccisumtabhtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"cells":{"fill":{"color":[["#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC"],["#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC"],["#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC"],["#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#

In [533]:
#CotA Summary Table HTML Code
cotasumtabjson = cotasumtable.to_json()

cotasumtabhtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {cotasumtabjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(cotasumtabhtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"cells":{"fill":{"color":[["#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF"],["#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1

##### <center>*Midterm Grade Summary Over Time Line Chart HTML Conversion*</center>

In [534]:
#AED MT Line Chart HTML Code
aedmtlinejson = aedratelines.to_json()

aedmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {aedmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(aedmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[84.0,8.0,58.0,10.0,75.0,9.0,68.0,9.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[81.55,57.14,73.42,83.33,80.65,90.0,81.93,81.82],"type":"scatter"},{"customdata":[6.0,5.0,12.0,1.0,11.0,1.0,8.0,1.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spri

In [535]:
#ARCH MT Line Chart HTML Code
archmtlinejson = archratelines.to_json()

archmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {archmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(archmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[786.0,810.0,958.0,845.0,1085.0,897.0,1120.0,734.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[79.47,79.26,77.76,79.79,79.72,79.8,77.29,79.87],"type":"scatter"},{"customdata":[121.0,113.0,145.0,101.0,147.0,118.0,167.0,111.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spr

In [536]:
#ARCS MT Line Chart HTML Code
arcsmtlinejson = arcsratelines.to_json()

arcsmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {arcsmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(arcsmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[11.0,30.0,27.0,37.0,20.0,38.0,16.0,19.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[84.62,75.0,81.82,80.43,74.07,90.48,76.19,86.36],"type":"scatter"},{"customdata":[0.0,4.0,5.0,5.0,2.0,1.0,2.0,2.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spr

In [537]:
#CMGT MT Line Chart HTML Code
cmgtmtlinejson = cmgtratelines.to_json()

cmgtmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {cmgtmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(cmgtmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[156.0,288.0,172.0,304.0,155.0,304.0,177.0,350.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[77.23,76.19,80.75,77.95,76.35,77.35,79.73,80.28],"type":"scatter"},{"customdata":[21.0,51.0,24.0,52.0,27.0,49.0,25.0,54.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024"

In [538]:
#ID MT Line Chart HTML Code
idmtlinejson = idratelines.to_json()

idmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {idmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(idmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[301.0,240.0,296.0,269.0,233.0,263.0,252.0,249.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[83.38,77.92,80.22,78.43,81.75,78.98,79.0,80.32],"type":"scatter"},{"customdata":[29.0,25.0,34.0,41.0,29.0,31.0,30.0,37.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024",

In [539]:
#CCI MT Line Chart HTML Code
ccimtlinejson = cciratelines.to_json()

ccimtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {ccimtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(ccimtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[92.0,84.0,103.0,79.0,52.0,61.0,73.0,76.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[73.6,81.55,82.4,79.8,68.42,78.21,76.84,88.37],"type":"scatter"},{"customdata":[21.0,11.0,13.0,11.0,10.0,9.0,14.0,9.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024"

In [540]:
#COMM MT Line Chart HTML Code
commmtlinejson = commratelines.to_json()

commmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {commmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(commmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[1144.0,637.0,1020.0,504.0,1128.0,541.0,1159.0,574.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[78.52,79.62,79.69,75.34,78.01,79.09,79.93,80.06],"type":"scatter"},{"customdata":[166.0,88.0,128.0,100.0,176.0,71.0,151.0,72.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spr

In [541]:
#EMAT MT Line Chart HTML Code
ematmtlinejson = ematratelines.to_json()

ematmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {ematmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(ematmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[243.0,226.0,202.0,207.0,198.0,209.0,245.0,217.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[77.88,79.3,79.53,82.47,77.65,80.38,77.04,78.06],"type":"scatter"},{"customdata":[40.0,34.0,22.0,22.0,36.0,21.0,33.0,37.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024",

In [542]:
#MDJ Total Term MT Line Chart HTML Code
mdjtotalmtlinejson = mdjtotalrateline.to_json()

mdjtotalmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {mdjtotalmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(mdjtotalmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[1058.0,961.0,1139.0,893.0,1064.0,825.0,1018.0,765.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[76.17,80.22,81.01,78.82,77.33,77.25,79.53,79.19],"type":"scatter"},{"customdata":[171.0,117.0,123.0,120.0,167.0,120.0,144.0,113.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spri

In [543]:
#MDJ Below 20k MT Line Chart HTML Code
mdjunder20kmtlinejson = mdjunder20krateline.to_json()

mdjunder20kmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {mdjunder20kmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(mdjunder20kmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[17.0,32.0,19.0,15.0,12.0,18.0,23.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025"],"y":[89.47,82.05,100.0,78.95,60.0,90.0,95.83],"type":"scatter"},{"customdata":[2.0,3.0,0.0,2.0,3.0,1.0,1.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025"],"y":[10

In [544]:
#MDJ MT 20k-25k Line Chart HTML Code
mdj20kto25kmtlinejson = mdj20kto25kratelines.to_json()

mdj20kto25kmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {mdj20kto25kmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(mdj20kto25kmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[53.0,41.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2022","Spring 2023"],"y":[76.81,83.67],"type":"scatter"},{"customdata":[10.0,3.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2022","Spring 2023"],"y":[14.49,6.12],"type":"scatter"},{"customdata":[6.0,5.0],"hovertemplate":"\u003cb\u003eMT Not Reported Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students

In [545]:
#MDJ MT Above 25k Line Chart HTML Code
mdjabove25kmtlinejson = mdjabove25krateline.to_json()

mdjabove25kmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {mdjabove25kmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(mdjabove25kmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[23.0,42.0,25.0,33.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024"],"y":[76.67,84.0,75.76,84.62],"type":"scatter"},{"customdata":[4.0,2.0,4.0,2.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024"],"y":[13.33,4.0,12.12,5.13],"type":"scatter"},{"customdata":[3.0,6.0,4.0,4.0],"hovertemplate":"\u003cb\u003eMT Not Reported Deta

In [546]:
#VCD Term Total MT Line Chart HTML Code
vcdtotalmtlinejson = vcdtotalrateline.to_json()

vcdtotalmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {vcdtotalmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(vcdtotalmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[627.0,533.0,647.0,548.0,615.0,449.0,439.0,353.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[78.38,78.27,78.42,80.71,79.05,79.19,78.67,81.34],"type":"scatter"},{"customdata":[97.0,77.0,101.0,66.0,90.0,59.0,65.0,54.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","F

In [547]:
#VCD MT Under 20k Line Chart HTML Code
vcdunder20kmtlinejson = vcdunder20kratelines.to_json()

vcdunder20kmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {vcdunder20kmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(vcdunder20kmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[58.0,21.0,73.0,40.0,44.0,31.0,39.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[75.32,95.45,82.02,75.47,88.0,70.45,78.0],"type":"scatter"},{"customdata":[8.0,1.0,8.0,6.0,5.0,6.0,9.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y"

In [548]:
#VCD MT Above 20k Line Chart HTML Code
vcdabove20kmtlinejson = vcdabove20kratelines.to_json()

vcdabove20kmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {vcdabove20kmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(vcdabove20kmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[45.0,23.0,61.0,33.0,81.0,29.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Fall 2025"],"y":[77.59,95.83,78.21,82.5,82.65,87.88],"type":"scatter"},{"customdata":[9.0,0.0,9.0,2.0,10.0,1.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Fall 2025"],"y":[15.52,0.0,11.54,5.0,10.2,3.03],"type":"scat

In [549]:
#ART MT Line Chart HTML Code
artsubmtlinejson = artsubratelines.to_json()

artsubmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {artsubmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(artsubmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[131.0,121.0,161.0,113.0,148.0,105.0,145.0,93.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[78.92,83.45,83.85,80.14,77.08,75.0,78.38,76.86],"type":"scatter"},{"customdata":[13.0,14.0,18.0,16.0,21.0,17.0,18.0,13.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","

In [550]:
#ARTH MT Line Chart HTML Code
arthsubmtlinejson = arthsubratelines.to_json()

arthsubmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {arthsubmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(arthsubmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[505.0,507.0,521.0,558.0,595.0,482.0,473.0,448.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[75.49,79.47,80.53,81.82,81.51,80.07,76.29,78.18],"type":"scatter"},{"customdata":[75.0,68.0,61.0,67.0,74.0,56.0,79.0,92.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024"

In [551]:
#ARTS MT Line Chart HTML Code
artssubmtlinejson = artssubratelines.to_json()

artssubmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {artssubmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(artssubmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[197.0,213.0,190.0,187.0,224.0,252.0,242.0,227.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[76.06,79.18,79.5,80.95,74.17,81.82,79.87,80.78],"type":"scatter"},{"customdata":[32.0,27.0,29.0,26.0,35.0,25.0,33.0,35.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024",

In [552]:
#FDM MT Term Total Line Chart HTML Code
fdmtotalmtlinejson = fdmtotalrateline.to_json()

fdmtotalmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {fdmtotalmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(fdmtotalmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[2294.0,2092.0,2142.0,2140.0,2166.0,1851.0,2072.0,1873.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[79.1,78.97,79.1,79.0,78.51,77.94,78.57,79.3],"type":"scatter"},{"customdata":[301.0,293.0,293.0,289.0,311.0,294.0,303.0,322.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spri

In [553]:
#FDM MT Below 20k Line Chart HTML Code
fdmbelow20kmtlinejson = fdmbelow20kratelines.to_json()

fdmbelow20kmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {fdmbelow20kmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(fdmbelow20kmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[200.0,140.0,180.0,139.0,142.0,118.0,159.0,166.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[78.74,78.21,78.26,79.43,71.72,76.13,79.5,79.81],"type":"scatter"},{"customdata":[27.0,18.0,25.0,16.0,31.0,22.0,22.0,30.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024",

In [554]:
#FDM MT Above 20k Line Chart HTML Code
fdmabove20kmtlinejson = fdmabove20kratelines.to_json()

fdmabove20kmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {fdmabove20kmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(fdmabove20kmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[206.0,136.0,146.0,169.0,183.0,120.0,154.0,139.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[79.54,84.47,82.02,81.25,78.21,81.63,79.38,78.98],"type":"scatter"},{"customdata":[29.0,17.0,15.0,17.0,29.0,15.0,21.0,28.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024"

In [555]:
#MUS MT Line Chart HTML Code
mustotalmtlinejson = mustotalrateline.to_json()

mustotalmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {mustotalmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(mustotalmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[971.0,912.0,890.0,871.0,911.0,928.0,879.0,877.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[79.27,79.03,82.26,78.4,80.48,80.07,79.62,79.08],"type":"scatter"},{"customdata":[136.0,113.0,104.0,143.0,131.0,117.0,121.0,150.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 20

In [556]:
#MUS MT Below 20k Line Chart HTML Code
musbelow20kmtlinejson = musbelow20kratelines.to_json()

musbelow20kmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {musbelow20kmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(musbelow20kmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[34.0,46.0,42.0,41.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Spring 2023","Spring 2024","Spring 2025","Spring 2026"],"y":[85.0,76.67,84.0,77.36],"type":"scatter"},{"customdata":[1.0,8.0,5.0,8.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Spring 2023","Spring 2024","Spring 2025","Spring 2026"],"y":[2.5,13.33,10.0,15.09],"type":"scatter"},{"customdata":[5.0,6.0,3.0,4.0],"hovertemplate":"\u003cb\u003eMT Not Report

In [557]:
#MUS MT Above 20k Line Chart HTML Code
musabove20kmtlinejson = musabove20kratelines.to_json()

musabove20kmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {musabove20kmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(musabove20kmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[12.0,23.0,13.0,7.0,9.0,7.0,83.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Spring 2026"],"y":[92.31,85.19,92.86,77.78,100.0,63.64,79.05],"type":"scatter"},{"customdata":[0.0,2.0,0.0,2.0,0.0,3.0,15.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Spring 2026"],"y"

In [558]:
#DAN MT Line Chart HTML Code
dansubmtlinejson = dansubratelines.to_json()

dansubmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {dansubmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(dansubmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[26.0,10.0,26.0,10.0,24.0,20.0,15.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025"],"y":[74.29,76.92,86.67,66.67,96.0,74.07,93.75],"type":"scatter"},{"customdata":[2.0,1.0,2.0,3.0,1.0,4.0,1.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025"],"y":[5

In [559]:
#THEA MT Term Total Line Chart HTML Code
theatotalmtlinejson = theatotalrateline.to_json()

theatotalmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {theatotalmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(theatotalmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[694.0,519.0,344.0,230.0,342.0,231.0,239.0,149.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[80.42,79.85,74.95,74.43,79.91,78.57,78.1,75.63],"type":"scatter"},{"customdata":[87.0,73.0,64.0,42.0,38.0,33.0,37.0,27.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fal

In [560]:
#THEA MT Below 14k Line Chart HTML Code
theabelow14kmtlinejson = theabelow14kratelines.to_json()

theabelow14kmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {theabelow14kmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(theabelow14kmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[13.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2025"],"y":[76.47],"type":"scatter"},{"customdata":[4.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2025"],"y":[23.53],"type":"scatter"},{"customdata":[0.0],"hovertemplate":"\u003cb\u003eMT Not Reported Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002f

In [561]:
#THEA MT 14k to 20k Line Chart HTML Code
thea14kto20kmtlinejson = thea14kto20kratelines.to_json()

thea14kto20kmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {thea14kto20kmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(thea14kto20kmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[15.0,6.0,13.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Spring 2023","Spring 2024","Spring 2025"],"y":[83.33,54.55,76.47],"type":"scatter"},{"customdata":[3.0,4.0,1.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Spring 2023","Spring 2024","Spring 2025"],"y":[16.67,36.36,5.88],"type":"scatter"},{"customdata":[0.0,1.0,3.0],"hovertemplate":"\u003cb\u003eMT Not Reported Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %

In [562]:
#THEA MT 20k to 25k Line Chart HTML Code
thea20kto25kmtlinejson = thea20kto25kratelines.to_json()

thea20kto25kmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {thea20kto25kmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(thea20kto25kmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[17.0,15.0,8.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Spring 2023","Spring 2024","Spring 2025"],"y":[77.27,75.0,72.73],"type":"scatter"},{"customdata":[1.0,2.0,2.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Spring 2023","Spring 2024","Spring 2025"],"y":[4.55,10.0,18.18],"type":"scatter"},{"customdata":[4.0,3.0,1.0],"hovertemplate":"\u003cb\u003eMT Not Reported Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x

In [563]:
#THEA MT Above 25k Line Chart HTML Code
theaabove25kmtlinejson = theaabove25kratelines.to_json()

theaabove25kmtlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {theaabove25kmtlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(theaabove25kmtlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":[12.0],"hovertemplate":"\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"green","width":2},"name":"MT C or Higher","visible":true,"x":["Fall 2025"],"y":[80.0],"type":"scatter"},{"customdata":[3.0],"hovertemplate":"\u003cb\u003eMT C-, D, F, W Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"red","width":2},"name":"MT C-, D, F, W","visible":true,"x":["Fall 2025"],"y":[20.0],"type":"scatter"},{"customdata":[0.0],"hovertemplate":"\u003cb\u003eMT Not Reported Details\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003eRate: %{y}%\u003cbr\u003e# of Students: %{customdata} students\u003cextra\u003e\u003c\u002fex

##### <center>*Intervention Tables HTML Conversion*</center>

In [564]:
#CAED Interventions Table HTML Code
caedintertabjson = caedintertab.to_json()

caedintertabhtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {caedintertabjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(caedintertabhtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"cells":{"fill":{"color":[["#FFFFFF","#FFF1CC","#EFAB00"],["#FFFFFF","#FFF1CC","#EFAB00"],["#FFFFFF","#FFF1CC","#EFAB00"]]},"font":{"color":"black"},"line":{"color":"black"},"values":[[1,2," "],["CAED Major, FR or SO, CAED Course, midterm D+ or below, still registered","CAED Major, JR or SR, CAED Course, midterm F, still registered","Total"],[[84],[14],[98]]]},"header":{"fill":{"color":"#EFAB00"},"font":{"color":"black","weight":"bold"},"line":{"color":"black"},"values":["Group #","CAED Midterm Intervention - Student Criteria","# of Instances"]},"visible":true,"type":"table"}],"layout":{"font":{"family":"Segoe UI"},"height":280,"margin":{"b":20,"l":20,"r":20,"t":80},"title":{"font":{"color":"black","size":24,"weight":"bold"},"text":"\u003cb\u003eSpring 2026 CAED Advisor Intervention Table","x":0.5,"xanchor":"center"},"width":700,"template":{"data":{"histogram2dco

In [565]:
#CCCI Interventions Table HTML Code
cciintertabjson = cciintertab.to_json()

cciintertabhtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {cciintertabjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(cciintertabhtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"cells":{"fill":{"color":[["#FFFFFF","#FFF1CC","#EFAB00"],["#FFFFFF","#FFF1CC","#EFAB00"],["#FFFFFF","#FFF1CC","#EFAB00"]]},"font":{"color":"black"},"line":{"color":"black"},"values":[[1,2," "],["CCI Major, FR or SO, CAED Course, midterm D+ or Below, still registered","CCI Major, JR or SR, CAED Course, midterm F, still registered","Total"],[[72],[19],[91]]]},"header":{"fill":{"color":"#EFAB00"},"font":{"color":"black","weight":"bold"},"line":{"color":"black"},"values":["Group #","CCI Midterm Intervention - Student Criteria","# of Instances"]},"visible":true,"type":"table"}],"layout":{"font":{"family":"Segoe UI"},"height":280,"margin":{"b":20,"l":20,"r":20,"t":80},"title":{"font":{"color":"black","size":24,"weight":"bold"},"text":"\u003cb\u003eSpring 2026 CCI Advisor Intervention Table","x":0.5,"xanchor":"center"},"width":700,"template":{"data":{"histogram2dcontou

In [566]:
#CotA Interventions Table HTML Code
cotaintertabjson = cotaintertab.to_json()

cotaintertabhtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {cotaintertabjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(cotaintertabhtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"cells":{"fill":{"color":[["#FFFFFF","#FFF1CC","#EFAB00"],["#FFFFFF","#FFF1CC","#EFAB00"],["#FFFFFF","#FFF1CC","#EFAB00"]]},"font":{"color":"black"},"line":{"color":"black"},"values":[[1,2," "],["CotA Major, FR or SO, CotA Course, midterm D+ or Below, still registered","CotA Major, JR or SR, CotA Course, midterm F, still registered","Total"],[[268],[40],[308]]]},"header":{"fill":{"color":"#EFAB00"},"font":{"color":"white","weight":"bold"},"line":{"color":"black"},"values":["Group #","CotA Midterm Intervention - Student Criteria","# of Instances"]},"visible":true,"type":"table"}],"layout":{"font":{"family":"Segoe UI"},"height":280,"margin":{"b":20,"l":20,"r":20,"t":80},"title":{"font":{"color":"black","size":24,"weight":"bold"},"text":"\u003cb\u003eSpring 2026 CotA Advisor Intervention Table","x":0.5,"xanchor":"center"},"width":700,"template":{"data":{"histogram2d

##### <center>*Interventions Over Time HTML Conversion*</center>

In [567]:
#CAED Interventions Over Time HTML Code
caedinterlinejson = caedinterlines.to_json()

caedinterlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {caedinterlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(caedinterlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":["Criteria: MT Grade of C or below","Criteria: MT Grade of C or below","Criteria: MT Grade of C or below","Criteria: MT Grade of C or below","Criteria: MT Grade of C or below","Criteria: MT Grade of C or below","Criteria: MT Grade of C or below","Criteria: MT Grade of D+ or lower"],"hovertemplate":"\u003cb\u003eFR\u002fSO Interventions\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003e# of Interventions Needed: %{y}\u003cbr\u003e%{customdata}\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"#003976","width":2},"name":"FR\u002fSO Interventions Needed","x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[133,122,157,115,165,142,147,84],"type":"scatter"},{"hovertemplate":"\u003cb\u003eJR\u002fSR Interventions\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003e# of Interven

In [568]:
#CCI Interventions Over Time HTML Code
cciinterlinejson = cciinterlines.to_json()

cciinterlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {cciinterlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(cciinterlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":["Criteria: MT Grade of C or below","Criteria: MT Grade of C or below","Criteria: MT Grade of C or below","Criteria: MT Grade of C or below","Criteria: MT Grade of C or below","Criteria: MT Grade of C or below","Criteria: MT Grade of C or below","Criteria: MT Grade of D+ or lower"],"hovertemplate":"\u003cb\u003eFR\u002fSO Interventions\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003e# of Interventions Needed: %{y}\u003cbr\u003e%{customdata}\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"#003976","width":2},"name":"FR\u002fSO Interventions Needed","x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[209,155,210,145,181,127,180,72],"type":"scatter"},{"hovertemplate":"\u003cb\u003eJR\u002fSR Interventions\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003e# of Interven

In [569]:
#CotA Interventions Over Time HTML Code
cotainterlinejson = cotainterlines.to_json()

cotainterlinehtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {cotainterlinejson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(cotainterlinehtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"customdata":["Criteria: MT Grade of C or below","Criteria: MT Grade of C or below","Criteria: MT Grade of C or below","Criteria: MT Grade of C or below","Criteria: MT Grade of C or below","Criteria: MT Grade of C or below","Criteria: MT Grade of C or below","Criteria: MT Grade of D+ or lower"],"hovertemplate":"\u003cb\u003eFR\u002fSO Interventions\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003e# of Interventions Needed: %{y}\u003cbr\u003e%{customdata}\u003cextra\u003e\u003c\u002fextra\u003e","line":{"color":"#003976","width":2},"name":"FR\u002fSO Interventions Needed","x":["Fall 2022","Spring 2023","Fall 2023","Spring 2024","Fall 2024","Spring 2025","Fall 2025","Spring 2026"],"y":[424,392,442,404,406,363,417,268],"type":"scatter"},{"hovertemplate":"\u003cb\u003eJR\u002fSR Interventions\u003c\u002fb\u003e\u003cbr\u003eTerm: %{x}\u003cbr\u003e# of Interve

##### <center>*Midterm-to-Final Grades Heat Map HTML Conversion*</center>

In [570]:
#AED Midterm to Final Grade HTML Code
aedheatjson = aedheat.to_json()

aedheathtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {aedheatjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(aedheathtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"colorbar":{"title":{"font":{"color":"black"},"text":"\u003cb\u003eStudent Density\u003c\u002fb\u003e"}},"colorscale":[[0.0,"#013271"],[0.1,"#5E636E"],[0.25,"#9D9576"],[1.0,"#E7D150"]],"hovertemplate":"\u003cb\u003eSector Details\u003c\u002fb\u003e\u003cbr\u003eMid Term Grade: %{x}\u003cbr\u003eFinal Grade: %{y}\u003cbr\u003e# of Students: %{z}\u003cextra\u003e\u003c\u002fextra\u003e","visible":true,"x":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"xgap":1,"y":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"ygap":1,"z":[[3,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,1,0,0,1,0,0,0],[0,0,0,0,0,0,0,0,0,0,0],[2,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,1,1,0,0],[0,0,0,0,1,1,0,3,0,0,0],[0,1,0,0,0,1,0,0,0,0,0],[0,0,0,0,0,0,1,0,0,0,0],[0,0,0,0,0,1,0,1,1,1,0],[1,0,0,0,1,0,1,1,4,5,4],[0,0,0,0,1,0,0,0,1,4,29]],"zmax":26,"zmin":0,"type":"heatmap"},{"

In [571]:
#ARCH Midterm to Final Grade HTML Code
archheatjson = archheat.to_json()

archheathtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {archheatjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(archheathtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"colorbar":{"title":{"font":{"color":"black"},"text":"\u003cb\u003eStudent Density\u003c\u002fb\u003e"}},"colorscale":[[0.0,"#013271"],[0.1,"#5E636E"],[0.25,"#9D9576"],[1.0,"#E7D150"]],"hovertemplate":"\u003cb\u003eSector Details\u003c\u002fb\u003e\u003cbr\u003eMid Term Grade: %{x}\u003cbr\u003eFinal Grade: %{y}\u003cbr\u003e# of Students: %{z}\u003cextra\u003e\u003c\u002fextra\u003e","visible":true,"x":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"xgap":1,"y":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"ygap":1,"z":[[29,2,2,0,0,1,5,7,2,0,6],[3,3,0,2,3,2,1,1,0,0,0],[1,2,2,1,0,1,0,1,0,0,0],[1,2,1,0,1,2,1,0,0,0,0],[2,2,1,1,6,1,3,6,2,2,2],[0,0,1,4,1,4,2,1,2,3,3],[1,2,1,1,5,4,2,5,3,3,4],[3,0,0,3,4,6,12,24,9,12,18],[1,0,1,0,1,4,6,11,10,8,10],[0,0,0,0,3,1,5,8,15,15,36],[3,0,0,0,3,1,2,8,8,34,206]],"zmax":154,"zmin":0,"type"

In [572]:
#ARCS Midterm to Final Grade HTML Code
arcsheatjson = arcsheat.to_json()

arcsheathtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {arcsheatjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(arcsheathtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"colorbar":{"title":{"font":{"color":"black"},"text":"\u003cb\u003eStudent Density\u003c\u002fb\u003e"}},"colorscale":[[0.0,"#013271"],[0.1,"#5E636E"],[0.25,"#9D9576"],[1.0,"#E7D150"]],"hovertemplate":"\u003cb\u003eSector Details\u003c\u002fb\u003e\u003cbr\u003eMid Term Grade: %{x}\u003cbr\u003eFinal Grade: %{y}\u003cbr\u003e# of Students: %{z}\u003cextra\u003e\u003c\u002fextra\u003e","visible":true,"x":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"xgap":1,"y":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"ygap":1,"z":[[1,0,0,0,0,0,1,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0],[0,1,0,0,0,0,0,0,0,0,0],[0,0,0,0,1,1,0,0,0,1,0],[0,0,0,0,0,0,0,2,0,0,1],[0,0,0,0,0,0,0,0,1,2,1],[0,0,0,0,1,0,0,0,0,0,4]],"zmax":2,"zmin":0,"type":"heatmap"},{"co

In [573]:
#CMGT Midterm to Final Grade HTML Code
cmgtheatjson = cmgtheat.to_json()

cmgtheathtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {cmgtheatjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(cmgtheathtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"colorbar":{"title":{"font":{"color":"black"},"text":"\u003cb\u003eStudent Density\u003c\u002fb\u003e"}},"colorscale":[[0.0,"#013271"],[0.1,"#5E636E"],[0.25,"#9D9576"],[1.0,"#E7D150"]],"hovertemplate":"\u003cb\u003eSector Details\u003c\u002fb\u003e\u003cbr\u003eMid Term Grade: %{x}\u003cbr\u003eFinal Grade: %{y}\u003cbr\u003e# of Students: %{z}\u003cextra\u003e\u003c\u002fextra\u003e","visible":true,"x":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"xgap":1,"y":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"ygap":1,"z":[[7,0,0,0,0,0,0,1,0,0,2],[1,1,0,0,0,0,0,0,0,0,1],[1,0,0,0,1,0,0,0,0,0,0],[0,1,0,1,0,0,0,0,0,0,1],[0,2,0,1,2,1,0,1,0,0,2],[0,0,0,1,1,3,1,0,0,1,0],[0,0,0,1,3,0,2,0,0,3,0],[2,0,0,0,2,1,4,6,4,1,4],[1,0,0,1,0,2,3,4,1,3,1],[0,0,0,1,0,0,0,6,7,8,12],[1,0,0,0,1,1,0,0,3,13,58]],"zmax":53,"zmin":0,"type":"heatmap"},

In [574]:
#ID Midterm to Final Grade HTML Code
idheatjson = idheat.to_json()

idheathtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {idheatjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(idheathtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"colorbar":{"title":{"font":{"color":"black"},"text":"\u003cb\u003eStudent Density\u003c\u002fb\u003e"}},"colorscale":[[0.0,"#013271"],[0.1,"#5E636E"],[0.25,"#9D9576"],[1.0,"#E7D150"]],"hovertemplate":"\u003cb\u003eSector Details\u003c\u002fb\u003e\u003cbr\u003eMid Term Grade: %{x}\u003cbr\u003eFinal Grade: %{y}\u003cbr\u003e# of Students: %{z}\u003cextra\u003e\u003c\u002fextra\u003e","visible":true,"x":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"xgap":1,"y":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"ygap":1,"z":[[9,2,0,0,1,0,0,1,1,0,0],[1,1,1,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0],[2,0,1,0,1,1,0,0,1,0,0],[0,0,1,0,2,1,3,2,1,0,0],[0,1,2,1,0,1,1,1,2,0,0],[0,0,2,0,0,1,1,3,0,1,4],[1,0,0,1,2,3,1,7,3,4,4],[0,0,0,0,2,0,3,10,7,3,6],[0,0,0,0,0,0,0,5,6,14,13],[1,0,0,1,0,1,0,4,4,16,89]],"zmax":99,"zmin":0,"type":"heatmap"

In [575]:
#CCI Midterm to Final Grade HTML Code
cciheatjson = cciheat.to_json()

cciheathtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {cciheatjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(cciheathtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"colorbar":{"title":{"font":{"color":"black"},"text":"\u003cb\u003eStudent Density\u003c\u002fb\u003e"}},"colorscale":[[0.0,"#013271"],[0.1,"#5E636E"],[0.25,"#9D9576"],[1.0,"#E7D150"]],"hovertemplate":"\u003cb\u003eSector Details\u003c\u002fb\u003e\u003cbr\u003eMid Term Grade: %{x}\u003cbr\u003eFinal Grade: %{y}\u003cbr\u003e# of Students: %{z}\u003cextra\u003e\u003c\u002fextra\u003e","visible":true,"x":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"xgap":1,"y":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"ygap":1,"z":[[6,0,0,0,0,1,0,0,0,1,0],[1,0,0,1,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,1,0,0,1],[1,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,1,0],[0,0,1,0,0,0,0,0,0,0,0],[0,0,0,0,1,0,0,1,0,0,2],[1,0,0,0,1,0,1,0,0,0,1],[0,0,0,0,1,1,2,0,1,2,2],[0,0,0,0,1,0,1,1,1,4,1],[0,0,0,0,0,0,0,2,3,3,23]],"zmax":32,"zmin":0,"type":"heatmap"},{"

In [576]:
#COMM Midterm to Final Grade HTML Code
commheatjson = commheat.to_json()

commheathtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {commheatjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(commheathtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"colorbar":{"title":{"font":{"color":"black"},"text":"\u003cb\u003eStudent Density\u003c\u002fb\u003e"}},"colorscale":[[0.0,"#013271"],[0.1,"#5E636E"],[0.25,"#9D9576"],[1.0,"#E7D150"]],"hovertemplate":"\u003cb\u003eSector Details\u003c\u002fb\u003e\u003cbr\u003eMid Term Grade: %{x}\u003cbr\u003eFinal Grade: %{y}\u003cbr\u003e# of Students: %{z}\u003cextra\u003e\u003c\u002fextra\u003e","visible":true,"x":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"xgap":1,"y":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"ygap":1,"z":[[6,2,1,1,0,0,0,0,0,1,0],[0,0,0,0,0,0,0,0,1,1,0],[1,1,0,0,0,0,0,0,0,0,0],[0,1,0,0,0,0,0,2,0,0,0],[0,1,1,1,2,1,0,0,0,0,2],[0,0,0,0,3,1,0,0,1,0,2],[0,0,1,1,2,0,5,2,1,0,0],[0,1,1,2,0,0,4,7,2,1,2],[1,0,1,0,0,1,3,2,2,1,3],[0,0,0,0,0,1,4,4,2,4,10],[0,0,0,0,1,0,2,4,6,7,72]],"zmax":76,"zmin":0,"type":"heatmap"},{

In [577]:
#EMAT Midterm to Final Grade HTML Code
ematheatjson = ematheat.to_json()

ematheathtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {ematheatjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(ematheathtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"colorbar":{"title":{"font":{"color":"black"},"text":"\u003cb\u003eStudent Density\u003c\u002fb\u003e"}},"colorscale":[[0.0,"#013271"],[0.1,"#5E636E"],[0.25,"#9D9576"],[1.0,"#E7D150"]],"hovertemplate":"\u003cb\u003eSector Details\u003c\u002fb\u003e\u003cbr\u003eMid Term Grade: %{x}\u003cbr\u003eFinal Grade: %{y}\u003cbr\u003e# of Students: %{z}\u003cextra\u003e\u003c\u002fextra\u003e","visible":true,"x":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"xgap":1,"y":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"ygap":1,"z":[[12,1,0,1,0,1,0,1,1,0,1],[0,0,1,0,0,0,0,0,0,1,0],[0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0],[0,0,1,0,1,0,2,0,0,1,0],[0,0,0,0,1,1,1,1,1,0,1],[0,1,0,0,2,2,4,3,3,1,1],[2,0,1,1,2,2,5,6,2,2,3],[0,0,0,0,0,1,3,4,5,2,4],[0,0,0,0,1,0,1,2,3,6,15],[0,0,0,0,2,0,1,1,7,14,78]],"zmax":68,"zmin":0,"type":"heatmap"}

In [578]:
#MDJ Midterm to Final Grade HTML Code
mdjheatjson = mdjheat.to_json()

mdjheathtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {mdjheatjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(mdjheathtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"colorbar":{"title":{"font":{"color":"black"},"text":"\u003cb\u003eStudent Density\u003c\u002fb\u003e"}},"colorscale":[[0.0,"#013271"],[0.1,"#5E636E"],[0.25,"#9D9576"],[1.0,"#E7D150"]],"hovertemplate":"\u003cb\u003eSector Details\u003c\u002fb\u003e\u003cbr\u003eMid Term Grade: %{x}\u003cbr\u003eFinal Grade: %{y}\u003cbr\u003e# of Students: %{z}\u003cextra\u003e\u003c\u002fextra\u003e","visible":true,"x":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"xgap":1,"y":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"ygap":1,"z":[[34,5,3,1,5,1,2,1,3,1,4],[2,0,0,0,0,0,0,0,1,0,0],[0,1,1,0,0,0,1,0,0,0,0],[1,4,0,2,1,0,0,1,0,0,2],[4,7,5,3,8,1,3,6,1,1,3],[1,2,3,3,6,2,4,2,2,2,4],[3,2,0,0,7,6,11,10,4,5,7],[2,1,1,4,8,2,13,17,15,8,15],[1,0,1,0,1,2,7,12,11,10,18],[0,0,0,0,2,3,4,17,16,22,46],[1,0,1,0,1,3,3,14,20,48,250]],"zmax":248,"zmin":0,

In [579]:
#VCD Midterm to Final Grade HTML Code
vcdheatjson = vcdheat.to_json()

vcdheathtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {vcdheatjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(vcdheathtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"colorbar":{"title":{"font":{"color":"black"},"text":"\u003cb\u003eStudent Density\u003c\u002fb\u003e"}},"colorscale":[[0.0,"#013271"],[0.1,"#5E636E"],[0.25,"#9D9576"],[1.0,"#E7D150"]],"hovertemplate":"\u003cb\u003eSector Details\u003c\u002fb\u003e\u003cbr\u003eMid Term Grade: %{x}\u003cbr\u003eFinal Grade: %{y}\u003cbr\u003e# of Students: %{z}\u003cextra\u003e\u003c\u002fextra\u003e","visible":true,"x":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"xgap":1,"y":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"ygap":1,"z":[[16,3,1,0,6,0,0,1,0,1,5],[1,1,0,0,0,0,1,1,1,0,0],[0,2,0,1,2,0,1,0,0,1,1],[1,0,1,2,1,1,1,1,2,0,0],[2,1,0,0,1,1,1,1,0,1,1],[1,1,2,0,3,0,2,2,1,0,2],[1,3,0,0,2,4,4,5,2,5,4],[3,2,0,0,4,3,5,14,2,6,12],[1,0,1,2,2,3,4,10,6,7,5],[1,0,0,0,1,0,3,8,14,9,25],[1,0,0,0,2,0,3,12,12,16,133]],"zmax":193,"zmin":0,"type":"h

In [580]:
#ART Midterm to Final Grade HTML Code
artheatjson = artheat.to_json()

artheathtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {artheatjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(artheathtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"colorbar":{"title":{"font":{"color":"black"},"text":"\u003cb\u003eStudent Density\u003c\u002fb\u003e"}},"colorscale":[[0.0,"#013271"],[0.1,"#5E636E"],[0.25,"#9D9576"],[1.0,"#E7D150"]],"hovertemplate":"\u003cb\u003eSector Details\u003c\u002fb\u003e\u003cbr\u003eMid Term Grade: %{x}\u003cbr\u003eFinal Grade: %{y}\u003cbr\u003e# of Students: %{z}\u003cextra\u003e\u003c\u002fextra\u003e","visible":true,"x":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"xgap":1,"y":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"ygap":1,"z":[[4,0,1,0,0,0,0,2,1,0,1],[0,1,0,0,0,1,0,0,0,0,0],[0,0,0,0,1,0,0,0,0,0,0],[0,0,0,0,1,0,0,0,0,0,0],[1,1,0,0,2,0,2,1,0,0,3],[1,0,1,0,0,0,1,0,0,0,0],[0,1,0,0,1,1,1,0,0,0,1],[0,0,0,0,1,3,2,6,0,2,4],[0,0,1,1,0,1,0,4,2,5,3],[0,0,0,0,1,0,1,3,3,1,5],[2,1,0,0,0,0,2,3,2,6,46]],"zmax":39,"zmin":0,"type":"heatmap"},{"

In [581]:
#ARTH Midterm to Final Grade HTML Code
arthheatjson = arthheat.to_json()

arthheathtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {arthheatjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(arthheathtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"colorbar":{"title":{"font":{"color":"black"},"text":"\u003cb\u003eStudent Density\u003c\u002fb\u003e"}},"colorscale":[[0.0,"#013271"],[0.1,"#5E636E"],[0.25,"#9D9576"],[1.0,"#E7D150"]],"hovertemplate":"\u003cb\u003eSector Details\u003c\u002fb\u003e\u003cbr\u003eMid Term Grade: %{x}\u003cbr\u003eFinal Grade: %{y}\u003cbr\u003e# of Students: %{z}\u003cextra\u003e\u003c\u002fextra\u003e","visible":true,"x":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"xgap":1,"y":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"ygap":1,"z":[[10,3,3,2,1,0,0,1,0,0,1],[1,0,0,0,3,0,0,0,0,0,0],[0,0,0,0,1,0,0,0,0,0,0],[0,1,0,0,0,0,0,0,0,0,1],[0,1,0,0,3,1,1,2,0,0,1],[1,2,0,0,0,2,1,4,2,1,1],[1,0,2,4,3,2,4,1,1,0,2],[1,0,0,1,0,2,5,5,7,3,8],[0,1,0,0,0,2,2,7,6,2,8],[1,0,0,1,0,0,2,4,8,9,18],[2,1,0,0,1,0,5,10,7,16,94]],"zmax":111,"zmin":0,"type":"heatmap

In [582]:
#ARTS Midterm to Final Grade HTML Code
artsheatjson = artsheat.to_json()

artsheathtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {artsheatjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(artsheathtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"colorbar":{"title":{"font":{"color":"black"},"text":"\u003cb\u003eStudent Density\u003c\u002fb\u003e"}},"colorscale":[[0.0,"#013271"],[0.1,"#5E636E"],[0.25,"#9D9576"],[1.0,"#E7D150"]],"hovertemplate":"\u003cb\u003eSector Details\u003c\u002fb\u003e\u003cbr\u003eMid Term Grade: %{x}\u003cbr\u003eFinal Grade: %{y}\u003cbr\u003e# of Students: %{z}\u003cextra\u003e\u003c\u002fextra\u003e","visible":true,"x":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"xgap":1,"y":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"ygap":1,"z":[[4,0,0,0,0,1,0,0,0,0,1],[0,0,0,0,0,1,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0],[0,1,1,0,0,0,1,0,0,0,1],[4,0,0,0,2,0,0,3,2,0,1],[0,1,0,0,0,1,1,2,0,1,2],[1,2,1,1,0,1,1,3,4,0,4],[0,0,0,0,3,0,2,7,2,1,1],[0,0,0,2,0,1,1,3,0,2,5],[0,0,0,0,0,3,0,1,3,7,10],[0,0,0,0,0,0,3,6,4,4,55]],"zmax":45,"zmin":0,"type":"heatmap"},{

In [583]:
#FDM Midterm to Final Grade HTML Code
fdmheatjson = fdmheat.to_json()

fdmheathtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {fdmheatjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(fdmheathtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"colorbar":{"title":{"font":{"color":"black"},"text":"\u003cb\u003eStudent Density\u003c\u002fb\u003e"}},"colorscale":[[0.0,"#013271"],[0.1,"#5E636E"],[0.25,"#9D9576"],[1.0,"#E7D150"]],"hovertemplate":"\u003cb\u003eSector Details\u003c\u002fb\u003e\u003cbr\u003eMid Term Grade: %{x}\u003cbr\u003eFinal Grade: %{y}\u003cbr\u003e# of Students: %{z}\u003cextra\u003e\u003c\u002fextra\u003e","visible":true,"x":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"xgap":1,"y":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"ygap":1,"z":[[106,14,9,5,7,6,10,7,4,2,8],[8,4,3,2,4,1,3,5,5,1,3],[2,4,2,3,2,3,2,2,0,0,1],[5,2,2,4,6,4,7,3,1,3,3],[5,10,2,7,23,6,7,7,9,2,7],[4,6,3,4,7,6,6,11,4,6,10],[10,6,2,10,12,16,20,24,7,15,18],[2,4,4,7,29,16,19,76,20,20,44],[4,2,0,3,5,11,16,47,34,23,45],[3,4,0,2,4,2,18,30,50,99,109],[4,3,0,1,5,3,15,47,49,109,696]

In [584]:
#MUS Midterm to Final Grade HTML Code
musheatjson = musheat.to_json()

musheathtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {musheatjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(musheathtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"colorbar":{"title":{"font":{"color":"black"},"text":"\u003cb\u003eStudent Density\u003c\u002fb\u003e"}},"colorscale":[[0.0,"#013271"],[0.1,"#5E636E"],[0.25,"#9D9576"],[1.0,"#E7D150"]],"hovertemplate":"\u003cb\u003eSector Details\u003c\u002fb\u003e\u003cbr\u003eMid Term Grade: %{x}\u003cbr\u003eFinal Grade: %{y}\u003cbr\u003e# of Students: %{z}\u003cextra\u003e\u003c\u002fextra\u003e","visible":true,"x":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"xgap":1,"y":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"ygap":1,"z":[[14,2,0,1,1,0,0,2,1,0,2],[1,0,0,0,2,0,0,0,0,0,0],[0,0,0,1,0,0,0,0,0,0,0],[0,2,1,0,0,0,0,2,0,0,1],[0,1,0,0,7,0,0,0,1,0,2],[0,0,1,1,0,0,0,4,1,2,1],[0,2,0,1,1,3,1,4,3,2,0],[0,1,0,0,0,2,3,12,1,4,3],[0,0,0,0,0,0,1,5,3,4,8],[0,0,0,0,0,2,3,6,6,10,12],[0,0,0,0,0,0,1,4,6,19,79]],"zmax":90,"zmin":0,"type":"heatmap

In [585]:
#DAN Midterm to Final Grade HTML Code
danheatjson = danheat.to_json()

danheathtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {danheatjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(danheathtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"colorbar":{"title":{"font":{"color":"black"},"text":"\u003cb\u003eStudent Density\u003c\u002fb\u003e"}},"colorscale":[[0.0,"#013271"],[0.1,"#5E636E"],[0.25,"#9D9576"],[1.0,"#E7D150"]],"hovertemplate":"\u003cb\u003eSector Details\u003c\u002fb\u003e\u003cbr\u003eMid Term Grade: %{x}\u003cbr\u003eFinal Grade: %{y}\u003cbr\u003e# of Students: %{z}\u003cextra\u003e\u003c\u002fextra\u003e","visible":true,"x":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"xgap":1,"y":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"ygap":1,"z":[[1,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,1,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,1,0,0],[0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,1,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,1,2,0],[0,0,0,0,0,0,0,0,0,1,4]],"zmax":6,"zmin":0,"type":"heatmap"},{"co

In [586]:
#THEA Midterm to Final Grade HTML Code
theaheatjson = theaheat.to_json()

theaheathtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {theaheatjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(theaheathtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"colorbar":{"title":{"font":{"color":"black"},"text":"\u003cb\u003eStudent Density\u003c\u002fb\u003e"}},"colorscale":[[0.0,"#013271"],[0.1,"#5E636E"],[0.25,"#9D9576"],[1.0,"#E7D150"]],"hovertemplate":"\u003cb\u003eSector Details\u003c\u002fb\u003e\u003cbr\u003eMid Term Grade: %{x}\u003cbr\u003eFinal Grade: %{y}\u003cbr\u003e# of Students: %{z}\u003cextra\u003e\u003c\u002fextra\u003e","visible":true,"x":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"xgap":1,"y":["0.0","1.0","1.3","1.7","2.0","2.3","2.7","3.0","3.3","3.7","4.0"],"ygap":1,"z":[[15,3,2,0,1,0,1,2,1,0,1],[0,0,0,1,0,0,0,1,0,0,1],[0,1,0,0,0,1,1,0,0,1,0],[1,1,0,1,1,1,0,0,2,1,0],[0,1,1,2,1,0,1,0,1,0,1],[1,2,0,0,3,1,2,1,0,0,1],[0,0,0,0,1,1,4,2,1,0,1],[0,0,1,0,3,1,1,7,4,3,4],[0,0,0,0,3,0,0,6,5,7,5],[0,0,0,1,0,0,2,10,5,14,6],[0,0,0,1,0,0,0,2,7,14,78]],"zmax":159,"zmin":0,"type":"heatmap

In [587]:
#GPA Table HTML Code
gpatabjson = gpatab.to_json()

gpatabhtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {gpatabjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(gpatabhtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"cells":{"fill":{"color":[["#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF"],["#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF"],["#CCFFCC","#CCFFCC","#CCFFCC","#CCFFCC","#CCFFCC","#CCFFCC","#FFFFCC","#FFFFCC","#FFFFCC","#FFFFCC","#FFCCCC","#CCFFCC","#FFCCCC","#B1B1B1","#FFCCCC","#B1B1B1","#B1B1B1","#B1B1B1","#B1B1B1","#B1B1B1","#B1B1B1"]]},"font":{"color":"black"},"line":{"color":"black"},"values":[["A","A-","B+","B","B-","C+","C","C-","D+","D","F","S","U","W","SF","NF","AU","NaN","DR","IN","NR"],[4.0,3.7,3.3,3.0,2.7,2.3,2.0,1.7,1.3,1.0,0.0,4.0,0.

##### <center>*Excel File for PowerBI to Read*</center>

The next section will be focused on converting the HTML code above into an Excel file for reading by PowerBI. It is recommended that, upon the code being written, the row be copy and paste into a "Master" file, where each row is a different term. This can then be set as a universal filter on PowerBI so only the selected term's figures are shown.

In [588]:
#Turning all of the code above into variables for use in the Excel file
htmlcode = {
    "Semester" : [currentterm],
    "CAED MT Sum Table HTML" : [caedsumtabhtml],
    "CCI MT Sum Table HTML" : [ccisumtabhtml],
    "CotA MT Sum Table HTML" : [cotasumtabhtml],
    "AED MT Over Time HTML" : [aedmtlinehtml],
    "ARCH MT Over Time HTML" : [archmtlinehtml],
    "ARCS MT Over Time HTML" : [arcsmtlinehtml],
    "CMGT MT Over Time HTML" : [cmgtmtlinehtml],
    "ID MT Over Time HTML" : [idmtlinehtml],
    "CCI MT Over Time HTML" : [ccimtlinehtml],
    "COMM MT Over Time HTML" : [commmtlinehtml],
    "EMAT MT Over Time HTML" : [ematmtlinehtml],
    "MDJ MT Over Time HTML - Term Total" : [mdjtotalmtlinehtml],
    "MDJ MT Over Time HTML - Under 20k" : [mdjunder20kmtlinehtml],
    "MDJ MT Over Time HTML - 20k to 25k" : [mdj20kto25kmtlinehtml],
    "MDJ MT Over Time HTML - Above 25k" : [mdjabove25kmtlinehtml],
    "VCD MT Over Time HTML - Term Total" : [vcdtotalmtlinehtml],
    "VCD MT Over Time HTML - Under 20k" : [vcdunder20kmtlinehtml],
    "VCD MT Over Time HTM - Above 20k" : [vcdabove20kmtlinehtml],
    "ART MT Over Time HTML" : [artsubmtlinehtml],
    "ARTH MT Over Time HTML" : [arthsubmtlinehtml],
    "ARTS MT Over Time HTML" : [artssubmtlinehtml],
    "FDM MT Over Time HTML - Term Total" : [fdmtotalmtlinehtml],
    "FDM MT Over Time HTML - Under 20k" : [fdmbelow20kmtlinehtml],
    "FDM MT Over Time HTML - Above 20k" : [fdmabove20kmtlinehtml],
    "MUS MT Over Time HTML - Term Total" : [mustotalmtlinehtml],
    "MUS MT Over Time HTML - Under 20k" : [musbelow20kmtlinehtml],
    "MUS MT Over Time HTML - Above 20k" : [musabove20kmtlinehtml],
    "DAN MT Over Time HTML" : [dansubmtlinehtml],
    "THEA MT Over Time HTML - Term Total" : [theatotalmtlinehtml],
    "THEA MT Over Time HTML - Under 14k" : [theabelow14kmtlinehtml],
    "THEA MT Over Time HTM - 14k to 20k" : [thea14kto20kmtlinehtml],
    "THEA MT Over Time HTML - 20k to 25k" : [thea20kto25kmtlinehtml],
    "THEA MT Over Time HTML - Above 25k" : [theaabove25kmtlinehtml],
    "CAED Inter Table HTML" : [caedintertabhtml],
    "CCI Inter Table HTML" : [cciintertabhtml],
    "CotA Inter Table HTML" : [cotaintertabhtml],
    "CAED Inter Over Time HTML" : [caedinterlinehtml],
    "CCI Inter Over Time HTML" : [cciinterlinehtml],
    "CotA Inter Over Time HTML" : [cotainterlinehtml],
    "AED Heat Map HTML" : [aedheathtml],
    "ARCH Heat Map HTML" : [archheathtml],
    "ARCS Heat Map HTML" : [arcsheathtml],
    "CMGT Heat Map HTML" : [cmgtheathtml],
    "ID Heat Map HTML" : [idheathtml],
    "CCI Heat Map HTML" : [cciheathtml],
    "COMM Heat Map HTML" : [commheathtml],
    "EMAT Heat Map HTML" : [ematheathtml],
    "MDJ Heat Map HTML" : [mdjheathtml],
    "VCD Heat Map HTML" : [vcdheathtml],
    "ART Heat Map HTML" : [artheathtml],
    "ARTH Heat Map HTML" : [artsheathtml],
    "ARTS Heat Map HTML" : [arthheathtml],
    "FDM Heat Map HTML" : [fdmheathtml],
    "MUS Heat Map HTML" : [musheathtml],
    "DAN Heat Map HTML" : [danheathtml],
    "THEA Heat Map HTML" : [theaheathtml],
    "GPA Conversion Table HTML": [gpatabhtml]
}

htmlcode = pd.DataFrame(htmlcode)
htmlcode

,Semester,CAED MT Sum Table HTML,CCI MT Sum Table HTML,CotA MT Sum Table HTML,AED MT Over Time HTML,ARCH MT Over Time HTML,ARCS MT Over Time HTML,CMGT MT Over Time HTML,ID MT Over Time HTML,CCI MT Over Time HTML,...,MDJ Heat Map HTML,VCD Heat Map HTML,ART Heat Map HTML,ARTH Heat Map HTML,ARTS Heat Map HTML,FDM Heat Map HTML,MUS Heat Map HTML,DAN Heat Map HTML,THEA Heat Map HTML,GPA Conversion Table HTML
0,[202610],"\n<div id=""plotly-div""></div>\n\n<script src=""...","\n<div id=""plotly-div""></div>\n\n<script src=""...","\n<div id=""plotly-div""></div>\n\n<script src=""...","\n<div id=""plotly-div""></div>\n\n<script src=""...","\n<div id=""plotly-div""></div>\n\n<script src=""...","\n<div id=""plotly-div""></div>\n\n<script src=""...","\n<div id=""plotly-div""></div>\n\n<script src=""...","\n<div id=""plotly-div""></div>\n\n<script src=""...","\n<div id=""plotly-div""></div>\n\n<script src=""...",...,"\n<div id=""plotly-div""></div>\n\n<script src=""...","\n<div id=""plotly-div""></div>\n\n<script src=""...","\n<div id=""plotly-div""></div>\n\n<script src=""...","\n<div id=""plotly-div""></div>\n\n<script src=""...","\n<div id=""plotly-div""></div>\n\n<script src=""...","\n<div id=""plotly-div""></div>\n\n<script src=""...","\n<div id=""plotly-div""></div>\n\n<script src=""...","\n<div id=""plotly-div""></div>\n\n<script src=""...","\n<div id=""plotly-div""></div>\n\n<script src=""...","\n<div id=""plotly-div""></div>\n\n<script src=""..."


In [589]:
#Loading XLSX Writer to create the Excel file and sheets
htmlwriter = pd.ExcelWriter(f"{prettyterm} HTML Code for PowerBI.xlsx",
                            engine = "xlsxwriter")

htmlcode.to_excel(htmlwriter, index=False,sheet_name = "HTML Code")

In [590]:
#Closing the Book so it loads the data
htmlwriter.close()

#### <center>**Section 10 - Limitations & Future Implementation**</center>

##### <center>*PowerBI Design Philosophy*</center>

As there is not a natural place to put this in PowerBI, I wanted to briefly discuss the design philosophy with the PowerBI File. There are a few notes with the design idea I wanted to touch upon:
- I wanted to replicate the look of a web application that users can navigate to access midterm grade data. Putting myself in the shoes of a leader in the hub, I figure they would want to hone in on their content areas especially. However, based on past conversations with hub leaders on similar matters, they are curious about how other units are doing and, in turn, how their students are doing in those courses. As such, I wanted to allow opportunities for users to jump between units and colleges in the hub as smoothly as possible. This was the logic of the sidebar menu - it has options to jump between that college's screens, but also a quick switch to the other college's versions of the same page.
- I wanted to replicate the blue/gold design aesthetic that was done in the figures. I opted to make blue the dominant color and yellow the secondary color. Additionally, I worked to make the font for the entire file Segoe UI to make the figures as uniform as possible.
- When choosing how to post the figures, the logic for posting them was as follows:
  - MT Summary Tables/Line Chart - Originally, I thought about hosting these both on the same page, with a toggle to turn one table off and the other on. However, when I sketched it out, it felt cramped and unintuitive. I feel these serve different purposes - the former allows a snapshot of the current term, while the latter allows a look at historical data. As such, I opted to break them up into separate pages.
  - Heat Maps/GPA Conversion Tables - This map was originally by itself. When I showed the project to my wife for some feedback, she indicated that it would be nice to know how the GPAs equate to grades - this isn't something known if you look at just the PowerBI table, as you need the code to see that. As such, I just translated the information in the code over to a table for hosting.
  - Intervention Table/Excel Sheet/Line Chart - When I first drafted the page, I felt the page was very crowded. It provides the most information out of every page, and can be overwhelming. Despite this, I think it is important to keep them all together. Alone, these figures would sit on very empty pages and not convey much information. Putting them together though tells a complete story - users can see how the current term's interventions work, quickly access the spreadsheet to get details about who needs interventions, and see how the current term shapes up to past semesters.
- This may just be a personal decision, but I like having the border and shadow effect on my figures in PowerBI. While I acknowledge they can make the figures more restricting (especially the borders), I think they make the figures look nicer and pop more. 

##### <center>*Limitations of the Project*</center>

Through the creation of this project, some limitations were noted as hindering its ability to reach its full potential:
1. The Anonymization of Data - As this project is being pushed to GitHub, student data has been anonymized to maintain student confidentiality. However, this does limit the project's ability to be a viable product. I am unable to create an accurate representation of the advisor contact sheet since we are missing student information, the accuracy of effectiveness of interventions has been hindered since grade data has been randomized and the anonymization of courses does not allow us to identify courses that may have "special" requirements to pass (i.e. requires a higher minimum grade, so the intervention criteria for those students is higher than others).
2. PowerBI Limitations - While working on uploading the figures to PowerBI, I was unable to get the native Python application to work due to time constraints. Instead, I had to use the HTML Contact Visualization which reads HTML versions of code to create the figures. This worked fine, but ran into issues with the more complex line charts in the Midterm Grades Over Time section. Certain subjects had so many courses that the character count for the code exceeded the 32,767 character limit that Excel and PowerBI have. This resulted in the figures being broken up into more digestible pieces.
3. Comparative Analysis Limitations - In its current form, users need to click through the different menus to view line charts for different units. While this still permits comparative analysis, this limits the ability to view the content on the same screen. If I were to do this, I would need to create bookmarks for every subject combination - the amount of which is compounded by the breaking of subjects into smaller pieces. 
4. Uploading the PowerBI File - I am unable to upload the PowerBI file to my workspace. This is due to the HTML Content visualization - since it isn't a Microsoft Certified Visualization, it cannot be uploaded and shared to the web due to the IT department's security measures. However, it can live natively on my PC and shared via link with others, but this would add additional steps to accessing the data (keeping track of access, remembering where the file is, etc.) that hosting it on my workspace would skip.
5. Assumptions About Interventions - This is an issue focused more on the heat maps - it assumes that the interventions are the reason why a student ended up increasing their final grades. While a conversation with an advisor could lead to students finding resources that helps their performance, there may be others who find resources elsewhere or use different support systems (i.e. talking to instructor, joining a peer support group, getting tutoring). 
6. Single Filters - A major limiter is the ability to only filter by a single filter. This is due in part to the HTML visual, which limits our library usage to single filters. Something I'd like to have done with the heat map is add filters to highlight FR/SO students and JR/SR students. This could help see how those populations fluctuate, and also adjust the rectangle to highlight those students who may qualify for an intervention.

##### <center>*Future Implementation*</center>

This is definitely a project I would like to move forward with and keep using in the future. However, there are a few ways forward to proceed:
1. Utilizing Microsoft Certified Visualizations - As touched on before, the visualization I used lacks the certification, meaning it cannot be posted to my workspace. If I can find a visualization that has this trait, I could post it to my workspace and use it more regularly.
2. Switching to the Python Visualization - I did attempt to use the native PowerBI Python visualization, but struggled to get it to work. I would like to spend more time researching this visualization and how it works - if I am able to get it to work, I can combo it with native PowerBI slicers to allow for filtering natively in PowerBI, while the Python visualization does the work of adjusting the figure based on filters. This may involve switching the visuals to a different language (Seaborn, Matplotlib, etc.), but I would like to keep it in plotly if possible due to it's customizability. 
3. Building it all in PowerBI - If the above isn't possible, the last resort would be breaking this file into two pieces. The data analysis/dataframe creation could occur via Python script, while the figure creation occurs in PowerBI. The only figure that may be lost in this process is the Heat Map - I was unable to find a native heat map visualization in PowerBI, so I would need to either create one using the Matrix visualization or find a certified pre-built one online. This could also help promote comparative analysis by making a version of the page with two figures (i.e. two line charts, two heat maps) and see how things change over time.
4. Additional Data - Adding additional qualifiers to the Intervention Contact Sheet would help enrich the data for the file. This could include checkboxes that indicate what resources students were directed to. This could then be used to see how well student's performance increased once final grades posted and, in cases where there is a significant jump, identify those students and see what resources they were directed to. This could help inform future intervention efforts and faculty could update their syllabus to include these resources to try and help these students earlier in the term.
5. Move to Dash - I believe the best way to improve this project is move the whole thing to Dash and create a "Data Hub" for the college hub. This would allow for greater interactivity, and less figures (i.e. making a single line chart that has filters for subject and course, rather than multiple figures for each subject). This could also allow for further comparative analysis, as we could host multiple figures on the same page. This will involve getting buy-in from IT to create/host a server for this data, as well as retain staff who can update/manage this information.

##### <center>*Final Thoughts*</center>

I wanted to close by offering my thoughts on how the implementation may go. While I think creating the figures in Dash and hosting a server is the best choice, I think the most realistic choice is proceeding with PowerBI dashboards. The institution is currently shifting toward hosting our data in PowerBI and more staff are becoming familiar with using the software. Compounding this line of thinking, I think there are concerns about retaining and/or recruiting staff who know how to code and maintain this system in Dash. Creating a homegrown application is great, but without the guarantee of maintaining staff who can oversee the process makes me cautious about moving in that direction.